# Layer 2 — Turning-Point Detection with MultiScaleCNN (Chapter 4.3)

**Thesis section.** 4.3 — local top / bottom detection, meta-labelled

**Inputs.** `data/tp_data/<TICKER>_{top,bottom}_{train,val,test}.npz`

**Outputs.** `checkpoints/multiscale_cnn_{top,bottom}.pt`, `results/meta_label_v2_results.json`, `results/turning_point_summary.md`

**Expected runtime.** 45–90 min. **Expected GPU.** T4 minimum.

> All paths in the CONFIG cell below resolve relative to the repo root. On
> Google Colab, uncomment the Drive fallback line.


In [ ]:
# === CONFIG (edit paths here) ===
from pathlib import Path

CONFIG = {
    "data_dir":        Path("../../data"),         # processed + features + splits
    "results_dir":     Path("../../results"),
    "checkpoints_dir": Path("../../checkpoints"),
    "seed":            42,
    "device":          "cuda",                      # or "cpu"
    # Colab fallback — uncomment if running on Colab with the dataset mounted:
    # "data_dir": Path("/content/drive/MyDrive/thesis_data"),
}
for key, path in CONFIG.items():
    if isinstance(path, Path):
        path.mkdir(parents=True, exist_ok=True)


In [ ]:
"""
======================================================================
  TURNING POINT — MODEL COMPARISON
  Config: rev=1.0%, dur=60, h=3 (45min)
  Tickers: ['AAPL', 'MSFT', 'GOOGL', 'GOOG', 'NVDA', 'TSLA', 'SPY', 'QQQ']
======================================================================

[1] Loading data & detecting turning points...
  Resampled: 574,498 (1min) → 43,340 (15min)
  AAPL: 1894 TPs, 388 in test
  Resampled: 497,673 (1min) → 42,907 (15min)
  MSFT: 1700 TPs, 379 in test
  Resampled: 403,977 (1min) → 39,056 (15min)
  GOOGL: 1698 TPs, 394 in test
  Resampled: 372,079 (1min) → 38,129 (15min)
  GOOG: 1654 TPs, 403 in test
  Resampled: 511,102 (1min) → 42,662 (15min)
  NVDA: 3239 TPs, 682 in test
  Resampled: 615,969 (1min) → 43,399 (15min)
  TSLA: 4276 TPs, 764 in test
  Resampled: 532,419 (1min) → 43,233 (15min)
  SPY: 921 TPs, 224 in test
  Resampled: 560,063 (1min) → 43,332 (15min)
  QQQ: 1282 TPs, 320 in test

[2] Building datasets (horizon=3)...
  Train: 123,235 | Val: 21,744
  TP in train: 8285 (6.7%)
  Features: 19

[3] Training & evaluating models...

  ──────────────────────────────────────────────────
  LSTM
  ──────────────────────────────────────────────────
    LSTM: epochs=16, val_loss=0.6661, time=168.1s
    AAPL  : TP=54.2% (N= 277, z=+1.38)
    MSFT  : TP=50.4% (N= 272, z=+0.12)
    GOOGL : TP=57.1% (N= 301, z=+2.48) ***
    GOOG  : TP=54.7% (N= 316, z=+1.69)
    NVDA  : TP=49.6% (N= 554, z=-0.17)
    TSLA  : TP=49.0% (N= 600, z=-0.49)
    SPY   : TP=55.4% (N= 139, z=+1.27)
    QQQ   : TP=53.2% (N= 235, z=+0.98)
    COMBINED: TP=52.1% (N=2694, z=+2.16, p=0.0309) ***

  ──────────────────────────────────────────────────
  Attn-LSTM
  ──────────────────────────────────────────────────
    Attn-LSTM: epochs=20, val_loss=0.6637, time=244.8s
    AAPL  : TP=54.9% (N= 277, z=+1.62)
    MSFT  : TP=51.8% (N= 272, z=+0.61)
    GOOGL : TP=55.1% (N= 301, z=+1.79)
    GOOG  : TP=56.3% (N= 316, z=+2.25) ***
    NVDA  : TP=53.1% (N= 554, z=+1.44)
    TSLA  : TP=51.5% (N= 600, z=+0.73)
    SPY   : TP=54.0% (N= 139, z=+0.93)
    QQQ   : TP=51.5% (N= 235, z=+0.46)
    COMBINED: TP=53.2% (N=2694, z=+3.35, p=0.0008) ***

  ──────────────────────────────────────────────────
  CNN-LSTM
  ──────────────────────────────────────────────────
    CNN-LSTM: epochs=17, val_loss=0.6642, time=239.9s
    AAPL  : TP=54.2% (N= 277, z=+1.38)
    MSFT  : TP=50.0% (N= 272, z=+0.00)
    GOOGL : TP=56.8% (N= 301, z=+2.36) ***
    GOOG  : TP=53.8% (N= 316, z=+1.35)
    NVDA  : TP=49.1% (N= 554, z=-0.42)
    TSLA  : TP=49.2% (N= 600, z=-0.41)
    SPY   : TP=55.4% (N= 139, z=+1.27)
    QQQ   : TP=52.8% (N= 235, z=+0.85)
    COMBINED: TP=51.7% (N=2694, z=+1.81, p=0.0701)

  ──────────────────────────────────────────────────
  Transformer
  ──────────────────────────────────────────────────
    Transformer: epochs=21, val_loss=0.6641, time=406.9s
    AAPL  : TP=54.9% (N= 277, z=+1.62)
    MSFT  : TP=50.7% (N= 272, z=+0.24)
    GOOGL : TP=56.1% (N= 301, z=+2.13) ***
    GOOG  : TP=53.8% (N= 316, z=+1.35)
    NVDA  : TP=49.6% (N= 554, z=-0.17)
    TSLA  : TP=49.3% (N= 600, z=-0.33)
    SPY   : TP=54.7% (N= 139, z=+1.10)
    QQQ   : TP=53.2% (N= 235, z=+0.98)
    COMBINED: TP=52.0% (N=2694, z=+2.08, p=0.0375) ***

  ──────────────────────────────────────────────────
  LightGBM
  ──────────────────────────────────────────────────
    LightGBM: iters=22, val_loss=0.6943, time=1.2s
    AAPL  : TP=49.1% (N= 277, z=-0.30)
    MSFT  : TP=49.3% (N= 272, z=-0.24)
    GOOGL : TP=52.5% (N= 301, z=+0.86)
    GOOG  : TP=55.7% (N= 316, z=+2.03) ***
    NVDA  : TP=51.6% (N= 554, z=+0.76)
    TSLA  : TP=49.7% (N= 600, z=-0.16)
    SPY   : TP=53.2% (N= 139, z=+0.76)
    QQQ   : TP=50.6% (N= 235, z=+0.20)
    COMBINED: TP=51.2% (N=2694, z=+1.27, p=0.2035)

======================================================================
  MODEL COMPARISON SUMMARY (rev=1.0%, dur=60, h=3)
======================================================================
  Model            TP_Acc      N       Z        p  Sig
  --------------------------------------------------
  Attn-LSTM         53.2%   2694   +3.35   0.0008 ***
  LSTM              52.1%   2694   +2.16   0.0309 ***
  Transformer       52.0%   2694   +2.08   0.0375 ***
  CNN-LSTM          51.7%   2694   +1.81   0.0701   *
  LightGBM          51.2%   2694   +1.27   0.2035

======================================================================
  BONUS: Also testing on rev=1.5%, dur=90, h=1
======================================================================
  Resampled: 574,498 (1min) → 43,340 (15min)
  Resampled: 497,673 (1min) → 42,907 (15min)
  Resampled: 403,977 (1min) → 39,056 (15min)
  Resampled: 372,079 (1min) → 38,129 (15min)
  Resampled: 511,102 (1min) → 42,662 (15min)
  Resampled: 615,969 (1min) → 43,399 (15min)
  Resampled: 532,419 (1min) → 43,233 (15min)
  Resampled: 560,063 (1min) → 43,332 (15min)

  LSTM:
    LSTM: epochs=17, val_loss=0.6775, time=119.9s
    AAPL  : TP=56.5% (N= 115, z=+1.40)
    MSFT  : TP=51.8% (N= 110, z=+0.38)
    GOOGL : TP=59.6% (N= 109, z=+2.01) ***
    GOOG  : TP=52.1% (N= 119, z=+0.46)
    NVDA  : TP=49.8% (N= 309, z=-0.06)
    TSLA  : TP=50.3% (N= 322, z=+0.11)
    SPY   : TP=55.3% (N=  47, z=+0.73)
    QQQ   : TP=57.5% (N=  80, z=+1.34)
    COMBINED: TP=52.5% (N=1211, z=+1.75, p=0.0796)

  Attn-LSTM:
    Attn-LSTM: epochs=19, val_loss=0.6770, time=160.0s
    AAPL  : TP=50.4% (N= 115, z=+0.09)
    MSFT  : TP=53.6% (N= 110, z=+0.76)
    GOOGL : TP=57.8% (N= 109, z=+1.63)
    GOOG  : TP=48.7% (N= 119, z=-0.28)
    NVDA  : TP=53.7% (N= 309, z=+1.31)
    TSLA  : TP=48.1% (N= 322, z=-0.67)
    SPY   : TP=44.7% (N=  47, z=-0.73)
    QQQ   : TP=51.2% (N=  80, z=+0.22)
    COMBINED: TP=51.2% (N=1211, z=+0.83, p=0.4046)

  CNN-LSTM:
    CNN-LSTM: epochs=19, val_loss=0.6772, time=175.3s
    AAPL  : TP=53.0% (N= 115, z=+0.65)
    MSFT  : TP=54.5% (N= 110, z=+0.95)
    GOOGL : TP=56.0% (N= 109, z=+1.25)
    GOOG  : TP=51.3% (N= 119, z=+0.28)
    NVDA  : TP=49.2% (N= 309, z=-0.28)
    TSLA  : TP=50.6% (N= 322, z=+0.22)
    SPY   : TP=44.7% (N=  47, z=-0.73)
    QQQ   : TP=55.0% (N=  80, z=+0.89)
    COMBINED: TP=51.3% (N=1211, z=+0.89, p=0.3730)

  Transformer:
---------------------------------------------------------------------------
KeyboardInterrupt                         Traceback (most recent call last)
/tmp/ipython-input-3533129308.py in <cell line: 0>()
    681
    682 if __name__ == "__main__":
--> 683     main()

4 frames
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py in _engine_run_backward(t_outputs, *args, **kwargs)
    839         unregister_hooks = _register_logging_hooks_on_whole_graph(t_outputs)
    840     try:
--> 841         return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
    842             t_outputs, *args, **kwargs
    843         )  # Calls into the C++ engine to run the backward pass

KeyboardInterrupt:
"""

In [ ]:
pip install lightgbm

In [ ]:
"""
Turning Point Model Comparison
==============================
Fixed config: 1.0% reversal, 60-bar min_dur, horizon=3 (45min)
Models: LSTM, Attention-LSTM, CNN-LSTM, Transformer, LightGBM

Reuses data pipeline from TurningPoint_LSTM.py
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
import time

# ============================================================
# 1. Config
# ============================================================
class Config:
    TICKERS = ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"]
    FREQ_RAW = "1min"
    FREQ_PRED = "15min"
    DRIVE_BASE = CONFIG['data_dir'] / r'splits'

    # Fixed best TP params
    TP_REVERSAL_PCT = 1.0
    TP_MIN_DURATION = 60
    RESAMPLE_PERIOD = 15
    HORIZON = 3  # 45min

    # Training
    SEQ_LEN = 30
    BATCH_SIZE = 32
    EPOCHS = 80
    LR = 5e-4
    PATIENCE = 15
    MIN_MOVE_PCT = 0.15

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    SEED = 42


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# 2. Import data pipeline from main script
# ============================================================
# Functions already defined in this notebook:
# detect_turning_points_causal, resample_to_15min, map_tp_to_15min,
# build_features, build_classification_dataset, load_ticker_data


# ============================================================
# 3. Model Definitions
# ============================================================

# --- Model 1: Baseline LSTM (same as before) ---
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)


# --- Model 2: Attention-LSTM ---
class AttentionLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.Tanh(),
            nn.Linear(hidden_size // 2, 1)
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)  # (B, T, H)
        # Attention weights
        attn_scores = self.attention(lstm_out).squeeze(-1)  # (B, T)
        attn_weights = torch.softmax(attn_scores, dim=1)    # (B, T)
        # Weighted sum
        context = torch.bmm(attn_weights.unsqueeze(1), lstm_out).squeeze(1)  # (B, H)
        return self.fc(context).squeeze(-1)


# --- Model 3: CNN-LSTM ---
class CNNLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        # CNN extracts local patterns
        self.conv1 = nn.Conv1d(input_size, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.bn2 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)

        # LSTM on CNN features
        self.lstm = nn.LSTM(64, hidden_size, num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        # x: (B, T, F) -> conv needs (B, F, T)
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)  # (B, 64, T//2)
        x = x.permute(0, 2, 1)  # (B, T//2, 64)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)


# --- Model 4: Transformer ---
class TransformerClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2, nhead=4):
        super().__init__()
        self.input_proj = nn.Linear(input_size, hidden_size)
        self.pos_encoding = nn.Parameter(torch.randn(1, 200, hidden_size) * 0.01)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size, nhead=nhead,
            dim_feedforward=hidden_size * 2,
            dropout=dropout, batch_first=True,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.fc = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

        # Causal mask
        self._causal_mask = None

    def _get_causal_mask(self, seq_len, device):
        if self._causal_mask is None or self._causal_mask.size(0) != seq_len:
            self._causal_mask = torch.triu(
                torch.ones(seq_len, seq_len, device=device) * float('-inf'),
                diagonal=1
            )
        return self._causal_mask

    def forward(self, x):
        B, T, F = x.shape
        x = self.input_proj(x) + self.pos_encoding[:, :T, :]
        mask = self._get_causal_mask(T, x.device)
        x = self.transformer(x, mask=mask)
        return self.fc(x[:, -1, :]).squeeze(-1)


# ============================================================
# 4. Generic trainer (works for all nn.Module models)
# ============================================================
def train_nn_model(model, X_train, y_train, X_val, y_val,
                   is_tp_train, config, model_name="Model"):
    model = model.to(config.DEVICE)

    n_pos = (y_train == 1).sum()
    n_neg = (y_train == 0).sum()
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(config.DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    sample_weights = np.ones(len(y_train), dtype=np.float32)
    sample_weights[is_tp_train] = 3.0

    optimizer = torch.optim.Adam(model.parameters(), lr=config.LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )

    train_ds = TensorDataset(
        torch.FloatTensor(X_train),
        torch.FloatTensor(y_train.astype(np.float32)),
        torch.FloatTensor(sample_weights)
    )
    val_ds = TensorDataset(
        torch.FloatTensor(X_val),
        torch.FloatTensor(y_val.astype(np.float32))
    )
    train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=config.BATCH_SIZE)

    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None

    t0 = time.time()
    for epoch in range(config.EPOCHS):
        model.train()
        train_loss = 0
        for X_b, y_b, w_b in train_loader:
            X_b = X_b.to(config.DEVICE)
            y_b = y_b.to(config.DEVICE)
            w_b = w_b.to(config.DEVICE)

            logits = model(X_b)
            loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight, weight=w_b)(logits, y_b)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item() * len(y_b)

        train_loss /= len(train_ds)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                X_b = batch[0].to(config.DEVICE)
                y_b = batch[1].to(config.DEVICE)
                logits = model(X_b)
                loss = criterion(logits, y_b)
                val_loss += loss.item() * len(y_b)
        val_loss /= len(val_ds)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config.PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
        model = model.to(config.DEVICE)

    train_time = time.time() - t0
    print(f"    {model_name}: epochs={epoch+1}, val_loss={best_val_loss:.4f}, "
          f"time={train_time:.1f}s")
    return model


# ============================================================
# 5. LightGBM trainer
# ============================================================
def train_lightgbm(X_train, y_train, X_val, y_val, is_tp_train, config):
    try:
        import lightgbm as lgb
    except ImportError:
        print("    [SKIP] LightGBM not installed")
        return None

    t0 = time.time()

    # Flatten sequences: use last timestep + rolling stats
    def flatten_features(X):
        B, T, F = X.shape
        last = X[:, -1, :]                    # (B, F) last timestep
        mean = X.mean(axis=1)                  # (B, F) mean over window
        std = X.std(axis=1)                    # (B, F) std over window
        diff = X[:, -1, :] - X[:, 0, :]       # (B, F) change over window
        return np.concatenate([last, mean, std, diff], axis=1)

    X_tr_flat = flatten_features(X_train)
    X_val_flat = flatten_features(X_val)

    # Sample weights
    sample_weights = np.ones(len(y_train), dtype=np.float32)
    sample_weights[is_tp_train] = 3.0

    train_data = lgb.Dataset(X_tr_flat, label=y_train, weight=sample_weights)
    val_data = lgb.Dataset(X_val_flat, label=y_val, reference=train_data)

    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'learning_rate': 0.05,
        'num_leaves': 31,
        'max_depth': 6,
        'min_child_samples': 50,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 0.1,
        'verbose': -1,
        'seed': config.SEED
    }

    callbacks = [lgb.early_stopping(50, verbose=False)]
    model = lgb.train(params, train_data, num_boost_round=500,
                      valid_sets=[val_data], callbacks=callbacks)

    train_time = time.time() - t0
    print(f"    LightGBM: iters={model.best_iteration}, "
          f"val_loss={model.best_score['valid_0']['binary_logloss']:.4f}, "
          f"time={train_time:.1f}s")

    return model


# ============================================================
# 6. Unified evaluation
# ============================================================
def evaluate_model(model, X_test, y_test, is_tp_test, config, model_name="",
                   is_lgb=False):
    if is_lgb:
        # Flatten same way as training
        B, T, F = X_test.shape
        last = X_test[:, -1, :]
        mean = X_test.mean(axis=1)
        std = X_test.std(axis=1)
        diff = X_test[:, -1, :] - X_test[:, 0, :]
        X_flat = np.concatenate([last, mean, std, diff], axis=1)
        probs = model.predict(X_flat)
    else:
        model.eval()
        with torch.no_grad():
            logits_list = []
            for i in range(0, len(X_test), 512):
                batch = torch.FloatTensor(X_test[i:i+512]).to(config.DEVICE)
                logits_list.append(model(batch).cpu().numpy())
            logits = np.concatenate(logits_list)
        probs = 1 / (1 + np.exp(-logits))

    preds = (probs > 0.5).astype(int)
    tp_mask = is_tp_test.astype(bool)

    result = {
        'model': model_name,
        'overall_acc': np.mean(preds == y_test) * 100,
        'N': len(y_test)
    }

    if tp_mask.sum() > 0:
        tp_acc = np.mean(preds[tp_mask] == y_test[tp_mask]) * 100
        tp_n = int(tp_mask.sum())
        z = (tp_acc/100 - 0.5) / np.sqrt(0.25 / tp_n)
        p_val = 2 * (1 - stats.norm.cdf(abs(z)))
        result.update({'tp_acc': tp_acc, 'tp_n': tp_n, 'z': z, 'p': p_val})

        # High confidence
        for thr in [0.6, 0.65, 0.7]:
            hc_tp = (np.abs(probs - 0.5) > (thr - 0.5)) & tp_mask
            if hc_tp.sum() > 10:
                result[f'tp_acc_{int(thr*100)}'] = np.mean(preds[hc_tp] == y_test[hc_tp]) * 100
                result[f'tp_n_{int(thr*100)}'] = int(hc_tp.sum())
    else:
        result.update({'tp_acc': None, 'tp_n': 0, 'z': 0, 'p': 1.0})

    return result


# ============================================================
# 7. Main
# ============================================================
def main():
    config = Config()
    set_seed(config.SEED)

    print(f"\n{'='*70}")
    print(f"  TURNING POINT — MODEL COMPARISON")
    print(f"  Config: rev={config.TP_REVERSAL_PCT}%, dur={config.TP_MIN_DURATION}, "
          f"h={config.HORIZON} ({config.HORIZON*15}min)")
    print(f"  Tickers: {config.TICKERS}")
    print(f"{'='*70}")

    # ── Load & process data ──
    print(f"\n[1] Loading data & detecting turning points...")
    ticker_data = {}
    for ticker in config.TICKERS:
        data = load_ticker_data(ticker, config)
        if data is not None:
            ticker_data[ticker] = data

    ticker_processed = {}
    for ticker, data in ticker_data.items():
        df = data['df']
        prices_1min = df['close'].values.astype(float)
        tp_events = detect_turning_points_causal(prices_1min, config)
        if len(tp_events) < 10:
            continue

        df_15min = resample_to_15min(df, config)
        tp_15min = map_tp_to_15min(tp_events, df, df_15min)
        train_mask = df_15min['timestamp'] <= data['train_end_ts']
        train_end_idx = int(train_mask.sum())
        tp_test = sum(1 for tp in tp_15min if tp['bar_idx'] >= train_end_idx)
        feature_df = build_features(df_15min, tp_15min, config)

        ticker_processed[ticker] = {
            'df_15min': df_15min, 'tp_15min': tp_15min,
            'feature_df': feature_df, 'train_end_ts': data['train_end_ts'],
            'train_end_idx': train_end_idx
        }
        print(f"  {ticker}: {len(tp_events)} TPs, {tp_test} in test")

    # ── Build datasets ──
    print(f"\n[2] Building datasets (horizon={config.HORIZON})...")
    all_X_train, all_y_train, all_is_tp_train = [], [], []
    all_X_val, all_y_val = [], []
    eval_test_data = {}

    for ticker, proc in ticker_processed.items():
        X, y, is_tp, timestamps = build_classification_dataset(
            proc['feature_df'], proc['df_15min'],
            proc['tp_15min'], config, horizon=config.HORIZON
        )
        if len(y) == 0:
            continue

        split_mask = timestamps <= np.datetime64(proc['train_end_ts'])
        X_tr, y_tr = X[split_mask], y[split_mask]
        is_tp_tr = is_tp[split_mask]
        X_te, y_te = X[~split_mask], y[~split_mask]
        is_tp_te = is_tp[~split_mask]

        val_size = max(int(len(X_tr) * 0.15), 1)
        all_X_train.append(X_tr[:-val_size])
        all_y_train.append(y_tr[:-val_size])
        all_is_tp_train.append(is_tp_tr[:-val_size])
        all_X_val.append(X_tr[-val_size:])
        all_y_val.append(y_tr[-val_size:])

        if is_tp_te.sum() > 0:
            eval_test_data[ticker] = {
                'X_test': X_te, 'y_test': y_te, 'is_tp_test': is_tp_te
            }

    X_train = np.concatenate(all_X_train)
    y_train = np.concatenate(all_y_train)
    is_tp_train = np.concatenate(all_is_tp_train)
    X_val = np.concatenate(all_X_val)
    y_val = np.concatenate(all_y_val)

    # Scale
    scaler = StandardScaler()
    n_tr, sl, nf = X_train.shape
    scaler.fit(X_train.reshape(-1, nf))
    X_train_s = scaler.transform(X_train.reshape(-1, nf)).reshape(X_train.shape)
    X_val_s = scaler.transform(X_val.reshape(-1, nf)).reshape(X_val.shape)
    for arr in [X_train_s, X_val_s]:
        np.nan_to_num(arr, copy=False, nan=0.0, posinf=3.0, neginf=-3.0)

    print(f"  Train: {len(X_train):,} | Val: {len(X_val):,}")
    print(f"  TP in train: {is_tp_train.sum()} ({is_tp_train.mean()*100:.1f}%)")
    print(f"  Features: {nf}")

    # ── Define models ──
    models_to_test = [
        ("LSTM", lambda: LSTMClassifier(nf, 64, 2, 0.2), False),
        ("Attn-LSTM", lambda: AttentionLSTM(nf, 64, 2, 0.2), False),
        ("CNN-LSTM", lambda: CNNLSTM(nf, 64, 2, 0.2), False),
        ("Transformer", lambda: TransformerClassifier(nf, 64, 2, 0.2, nhead=4), False),
        ("LightGBM", None, True),
    ]

    # ── Train & evaluate each model ──
    print(f"\n[3] Training & evaluating models...")
    all_results = []

    for model_name, model_fn, is_lgb in models_to_test:
        print(f"\n  {'─'*50}")
        print(f"  {model_name}")
        print(f"  {'─'*50}")

        set_seed(config.SEED)

        if is_lgb:
            model = train_lightgbm(X_train_s, y_train, X_val_s, y_val,
                                   is_tp_train, config)
            if model is None:
                continue
        else:
            model = model_fn()
            model = train_nn_model(model, X_train_s, y_train, X_val_s, y_val,
                                   is_tp_train, config, model_name)

        # Evaluate per-ticker
        combined_correct = 0
        combined_total = 0

        for ticker, tdata in eval_test_data.items():
            X_te = scaler.transform(
                tdata['X_test'].reshape(-1, nf)
            ).reshape(tdata['X_test'].shape)
            np.nan_to_num(X_te, copy=False, nan=0.0, posinf=3.0, neginf=-3.0)

            res = evaluate_model(model, X_te, tdata['y_test'],
                                 tdata['is_tp_test'], config,
                                 model_name=model_name, is_lgb=is_lgb)

            if res['tp_acc'] is not None:
                combined_correct += int(res['tp_acc']/100 * res['tp_n'])
                combined_total += res['tp_n']

                sig = '***' if res['p'] < 0.05 else '   '
                extra = ''
                if 'tp_acc_60' in res:
                    extra += f"  [>60%: {res['tp_acc_60']:.1f}% N={res['tp_n_60']}]"
                if 'tp_acc_70' in res:
                    extra += f"  [>70%: {res['tp_acc_70']:.1f}% N={res['tp_n_70']}]"
                print(f"    {ticker:6s}: TP={res['tp_acc']:.1f}% "
                      f"(N={res['tp_n']:4d}, z={res['z']:+.2f}) {sig}{extra}")

        if combined_total > 0:
            comb_acc = combined_correct / combined_total * 100
            z_c = (comb_acc/100 - 0.5) / np.sqrt(0.25 / combined_total)
            p_c = 2 * (1 - stats.norm.cdf(abs(z_c)))
            sig = '***' if p_c < 0.05 else ''
            print(f"    {'COMBINED':6s}: TP={comb_acc:.1f}% "
                  f"(N={combined_total}, z={z_c:+.2f}, p={p_c:.4f}) {sig}")

            all_results.append({
                'model': model_name, 'tp_acc': comb_acc,
                'tp_n': combined_total, 'z': z_c, 'p': p_c
            })

    # ── Summary ──
    print(f"\n{'='*70}")
    print(f"  MODEL COMPARISON SUMMARY (rev=1.0%, dur=60, h=3)")
    print(f"{'='*70}")
    print(f"  {'Model':<15s} {'TP_Acc':>7s} {'N':>6s} {'Z':>7s} {'p':>8s} {'Sig':>4s}")
    print(f"  {'-'*50}")
    for r in sorted(all_results, key=lambda x: -x['tp_acc']):
        sig = '***' if r['p'] < 0.05 else '  *' if r['p'] < 0.1 else '   '
        print(f"  {r['model']:<15s} {r['tp_acc']:6.1f}% {r['tp_n']:6d} "
              f"{r['z']:+7.2f} {r['p']:8.4f} {sig}")

    # Also test on 1.5%/90 h=1 (second-best config)
    print(f"\n{'='*70}")
    print(f"  BONUS: Also testing on rev=1.5%, dur=90, h=1")
    print(f"{'='*70}")

    config2 = Config()
    config2.TP_REVERSAL_PCT = 1.5
    config2.TP_MIN_DURATION = 90
    config2.HORIZON = 1

    # Re-detect TPs with new params
    ticker_processed2 = {}
    for ticker, data in ticker_data.items():
        df = data['df']
        prices_1min = df['close'].values.astype(float)
        tp_events = detect_turning_points_causal(prices_1min, config2)
        if len(tp_events) < 10:
            continue
        df_15min = resample_to_15min(df, config2)
        tp_15min = map_tp_to_15min(tp_events, df, df_15min)
        train_mask = df_15min['timestamp'] <= data['train_end_ts']
        train_end_idx = int(train_mask.sum())
        feature_df = build_features(df_15min, tp_15min, config2)
        ticker_processed2[ticker] = {
            'df_15min': df_15min, 'tp_15min': tp_15min,
            'feature_df': feature_df, 'train_end_ts': data['train_end_ts'],
            'train_end_idx': train_end_idx
        }

    # Build datasets for config2
    all_X_train2, all_y_train2, all_is_tp_train2 = [], [], []
    all_X_val2, all_y_val2 = [], []
    eval_test_data2 = {}

    for ticker, proc in ticker_processed2.items():
        X, y, is_tp, timestamps = build_classification_dataset(
            proc['feature_df'], proc['df_15min'],
            proc['tp_15min'], config2, horizon=config2.HORIZON
        )
        if len(y) == 0:
            continue
        split_mask = timestamps <= np.datetime64(proc['train_end_ts'])
        X_tr, y_tr = X[split_mask], y[split_mask]
        is_tp_tr = is_tp[split_mask]
        X_te, y_te = X[~split_mask], y[~split_mask]
        is_tp_te = is_tp[~split_mask]
        val_size = max(int(len(X_tr) * 0.15), 1)
        all_X_train2.append(X_tr[:-val_size])
        all_y_train2.append(y_tr[:-val_size])
        all_is_tp_train2.append(is_tp_tr[:-val_size])
        all_X_val2.append(X_tr[-val_size:])
        all_y_val2.append(y_tr[-val_size:])
        if is_tp_te.sum() > 0:
            eval_test_data2[ticker] = {
                'X_test': X_te, 'y_test': y_te, 'is_tp_test': is_tp_te
            }

    X_train2 = np.concatenate(all_X_train2)
    y_train2 = np.concatenate(all_y_train2)
    is_tp_train2 = np.concatenate(all_is_tp_train2)
    X_val2 = np.concatenate(all_X_val2)
    y_val2 = np.concatenate(all_y_val2)

    scaler2 = StandardScaler()
    scaler2.fit(X_train2.reshape(-1, nf))
    X_train2_s = scaler2.transform(X_train2.reshape(-1, nf)).reshape(X_train2.shape)
    X_val2_s = scaler2.transform(X_val2.reshape(-1, nf)).reshape(X_val2.shape)
    for arr in [X_train2_s, X_val2_s]:
        np.nan_to_num(arr, copy=False, nan=0.0, posinf=3.0, neginf=-3.0)

    bonus_results = []
    for model_name, model_fn, is_lgb in models_to_test:
        print(f"\n  {model_name}:")
        set_seed(config2.SEED)

        if is_lgb:
            m = train_lightgbm(X_train2_s, y_train2, X_val2_s, y_val2,
                               is_tp_train2, config2)
            if m is None:
                continue
        else:
            m = model_fn()
            m = train_nn_model(m, X_train2_s, y_train2, X_val2_s, y_val2,
                               is_tp_train2, config2, model_name)

        cc, ct = 0, 0
        for ticker, tdata in eval_test_data2.items():
            X_te = scaler2.transform(
                tdata['X_test'].reshape(-1, nf)
            ).reshape(tdata['X_test'].shape)
            np.nan_to_num(X_te, copy=False, nan=0.0, posinf=3.0, neginf=-3.0)
            res = evaluate_model(m, X_te, tdata['y_test'],
                                 tdata['is_tp_test'], config2,
                                 model_name=model_name, is_lgb=is_lgb)
            if res['tp_acc'] is not None:
                cc += int(res['tp_acc']/100 * res['tp_n'])
                ct += res['tp_n']
                sig = '***' if res['p'] < 0.05 else '   '
                print(f"    {ticker:6s}: TP={res['tp_acc']:.1f}% "
                      f"(N={res['tp_n']:4d}, z={res['z']:+.2f}) {sig}")

        if ct > 0:
            ca = cc / ct * 100
            zz = (ca/100 - 0.5) / np.sqrt(0.25 / ct)
            pp = 2 * (1 - stats.norm.cdf(abs(zz)))
            sig = '***' if pp < 0.05 else ''
            print(f"    {'COMBINED':6s}: TP={ca:.1f}% (N={ct}, z={zz:+.2f}, p={pp:.4f}) {sig}")
            bonus_results.append({
                'model': model_name, 'tp_acc': ca, 'tp_n': ct, 'z': zz, 'p': pp
            })

    print(f"\n  BONUS SUMMARY (rev=1.5%, dur=90, h=1)")
    print(f"  {'Model':<15s} {'TP_Acc':>7s} {'N':>6s} {'Z':>7s} {'p':>8s}")
    print(f"  {'-'*50}")
    for r in sorted(bonus_results, key=lambda x: -x['tp_acc']):
        sig = '***' if r['p'] < 0.05 else '  *' if r['p'] < 0.1 else '   '
        print(f"  {r['model']:<15s} {r['tp_acc']:6.1f}% {r['tp_n']:6d} "
              f"{r['z']:+7.2f} {r['p']:8.4f} {sig}")


if __name__ == "__main__":
    main()

In [ ]:
"""
CNN Trend Reversal v2 — Downtrend Focus + Fine Threshold Sweep
================================================================
Uses the SAME trained model from v2, just adds detailed evaluation.

Add this AFTER run_trend_reversal_v2() finishes, using the same model.
OR: paste the full v2 code first, then replace the main function with this.

Key changes:
1. Fine-grained threshold sweep: 0.45 to 0.65 in 0.025 steps
2. Separate down/up analysis
3. Per-ticker breakdown at best threshold
4. Profit simulation (assuming fixed % gain on correct, fixed % loss on wrong)
"""

def evaluate_detailed(model, X_test, y_test, dirs_test, config, label=""):
    """Fine-grained evaluation focusing on downtrend reversals."""
    model.eval()
    with torch.no_grad():
        out = []
        for i in range(0, len(X_test), 1024):
            b = torch.FloatTensor(X_test[i:i+1024]).to(config.DEVICE)
            out.append(torch.sigmoid(model(b)).cpu().numpy())
    probs = np.concatenate(out)
    dirs_arr = np.array(dirs_test)

    down_mask = dirs_arr == 'down'
    up_mask = dirs_arr == 'up'

    base_all = (y_test == 1).mean()
    base_down = y_test[down_mask].mean() if down_mask.sum() > 0 else 0
    base_up = y_test[up_mask].mean() if up_mask.sum() > 0 else 0

    print(f"\n  {label}")
    print(f"  Total: {len(y_test)} | Down: {down_mask.sum()} | Up: {up_mask.sum()}")
    print(f"  Base reversal rate: all={base_all*100:.1f}%, down={base_down*100:.1f}%, up={base_up*100:.1f}%")

    # ── Fine-grained sweep: DOWNTREND ONLY ──
    print(f"\n  ── DOWNTREND REVERSALS (buy signal) ──")
    print(f"  {'thr':>5s}  {'signals':>7s}  {'prec':>6s}  {'recall':>6s}  {'lift':>5s}  {'z':>6s}  {'p':>8s}")
    print(f"  {'─'*55}")

    best_thr, best_score = 0.5, 0
    for thr_x10 in range(45, 66):
        thr = thr_x10 / 100.0
        sig = (probs > thr) & down_mask
        n_sig = sig.sum()
        if n_sig < 10:
            continue

        prec = np.mean(y_test[sig] == 1) * 100
        rec = np.mean((probs[down_mask & (y_test == 1)] > thr)) * 100 if (down_mask & (y_test == 1)).sum() > 0 else 0
        lift = prec / (base_down * 100) if base_down > 0 else 0
        z = (prec / 100 - base_down) / np.sqrt(base_down * (1 - base_down) / n_sig) if base_down > 0 else 0
        p = 2 * (1 - stats.norm.cdf(abs(z)))
        s = '***' if p < 0.05 else '  *' if p < 0.1 else '   '

        # Score: balance precision and signal count
        # Want precision > 50% with decent N
        score = (prec - 50) * np.log(n_sig + 1) if prec > 50 else 0
        if score > best_score:
            best_score = score
            best_thr = thr

        print(f"  {thr:5.2f}  {n_sig:7d}  {prec:5.1f}%  {rec:5.1f}%  {lift:4.2f}x  {z:+5.2f}  {p:7.4f} {s}")

    # ── Same for UPTREND ──
    print(f"\n  ── UPTREND REVERSALS (sell signal) ──")
    print(f"  {'thr':>5s}  {'signals':>7s}  {'prec':>6s}  {'recall':>6s}  {'lift':>5s}  {'z':>6s}  {'p':>8s}")
    print(f"  {'─'*55}")

    for thr_x10 in range(45, 66):
        thr = thr_x10 / 100.0
        sig = (probs > thr) & up_mask
        n_sig = sig.sum()
        if n_sig < 10:
            continue

        prec = np.mean(y_test[sig] == 1) * 100
        rec = np.mean((probs[up_mask & (y_test == 1)] > thr)) * 100 if (up_mask & (y_test == 1)).sum() > 0 else 0
        lift = prec / (base_up * 100) if base_up > 0 else 0
        z = (prec / 100 - base_up) / np.sqrt(base_up * (1 - base_up) / n_sig) if base_up > 0 else 0
        p = 2 * (1 - stats.norm.cdf(abs(z)))
        s = '***' if p < 0.05 else '  *' if p < 0.1 else '   '

        print(f"  {thr:5.2f}  {n_sig:7d}  {prec:5.1f}%  {rec:5.1f}%  {lift:4.2f}x  {z:+5.2f}  {p:7.4f} {s}")

    # ── Best threshold per-ticker breakdown ──
    print(f"\n  ── PER-TICKER @ best downtrend thr={best_thr:.2f} ──")
    return best_thr, probs


def run_detailed_eval():
    """Run v2 training then detailed evaluation."""
    config = TrendRevConfig2()
    set_seed(config.SEED)

    print(f"\n{'='*70}")
    print(f"  CNN TREND REVERSAL v2 — DETAILED ANALYSIS")
    print(f"  Focus: Downtrend reversals (V-bottom detection)")
    print(f"{'='*70}")

    print(f"\n[1] Loading data...")

    all_X_tr, all_y_tr, all_d_tr = [], [], []
    all_X_v, all_y_v, all_d_v = [], [], []
    test_data = {}

    for ticker in config.TICKERS:
        data = load_ticker_data(ticker, config)
        if data is None:
            continue
        df = data['df']
        train_end_ts = data['train_end_ts']

        df_15min = resample_to_15min(df, config)
        features = build_bar_features_v2(df_15min)
        positions = find_trend_and_label(df_15min, config)

        if len(positions) < 50:
            continue

        train_mask = df_15min['timestamp'] <= train_end_ts
        train_end_idx = int(train_mask.sum())

        pos_train = [p for p in positions if p['bar_idx'] < train_end_idx]
        pos_test = [p for p in positions if p['bar_idx'] >= train_end_idx]

        X, y, dirs = build_dataset(features, positions, config)
        split = len(pos_train)
        X_tr, y_tr, d_tr = X[:split], y[:split], dirs[:split]
        X_te, y_te, d_te = X[split:], y[split:], dirs[split:]

        rev_tr = y_tr.mean() * 100 if len(y_tr) > 0 else 0
        rev_te = y_te.mean() * 100 if len(y_te) > 0 else 0

        n_down_te = sum(1 for d in d_te if d == 'down')
        n_up_te = sum(1 for d in d_te if d == 'up')

        print(f"  {ticker}: train={len(y_tr)}, test={len(y_te)} "
              f"(down={n_down_te}, up={n_up_te}, rev={rev_te:.1f}%)")

        if len(y_tr) > 50:
            vs = max(int(len(X_tr) * 0.15), 1)
            all_X_tr.append(X_tr[:-vs])
            all_y_tr.append(y_tr[:-vs])
            all_d_tr.extend(d_tr[:-vs])
            all_X_v.append(X_tr[-vs:])
            all_y_v.append(y_tr[-vs:])
            all_d_v.extend(d_tr[-vs:])

        if len(y_te) > 10:
            test_data[ticker] = {'X': X_te, 'y': y_te, 'dirs': d_te}

    X_train = np.concatenate(all_X_tr)
    y_train = np.concatenate(all_y_tr)
    X_val = np.concatenate(all_X_v)
    y_val = np.concatenate(all_y_v)

    print(f"\n  Combined: train={len(X_train):,}, val={len(X_val):,}")
    print(f"  Reversal rate: train={y_train.mean()*100:.1f}%, val={y_val.mean()*100:.1f}%")

    # ── Train ──
    print(f"\n[2] Training CNN...")
    set_seed(config.SEED)
    model = train_cnn(X_train, y_train, X_val, y_val, config)

    # ── Detailed Val ──
    print(f"\n[3] Detailed validation:")
    evaluate_detailed(model, X_val, y_val, all_d_v, config, "VALIDATION")

    # ── Detailed Test: Combined ──
    X_all = np.concatenate([td['X'] for td in test_data.values()])
    y_all = np.concatenate([td['y'] for td in test_data.values()])
    d_all = sum([td['dirs'] for td in test_data.values()], [])

    print(f"\n[4] Detailed test (combined):")
    best_thr, probs_all = evaluate_detailed(model, X_all, y_all, d_all, config, "TEST COMBINED")

    # ── Per-ticker at best threshold ──
    print(f"\n[5] Per-ticker @ thr={best_thr:.2f} (downtrend only):")
    print(f"  {'ticker':>6s}  {'signals':>7s}  {'prec':>6s}  {'base':>5s}  {'lift':>5s}")
    print(f"  {'─'*40}")

    for ticker, td in test_data.items():
        model.eval()
        with torch.no_grad():
            out = []
            for i in range(0, len(td['X']), 1024):
                b = torch.FloatTensor(td['X'][i:i+1024]).to(config.DEVICE)
                out.append(torch.sigmoid(model(b)).cpu().numpy())
        p = np.concatenate(out)
        da = np.array(td['dirs'])
        yt = td['y']

        down = da == 'down'
        sig = (p > best_thr) & down
        base = yt[down].mean() * 100 if down.sum() > 0 else 0

        if sig.sum() > 5:
            prec = np.mean(yt[sig] == 1) * 100
            lift = prec / base if base > 0 else 0
            print(f"  {ticker:>6s}  {sig.sum():7d}  {prec:5.1f}%  {base:4.1f}%  {lift:4.2f}x")

    # ── Profit simulation ──
    print(f"\n[6] Profit simulation @ thr={best_thr:.2f} (downtrend only):")
    dirs_arr = np.array(d_all)
    down_mask = dirs_arr == 'down'
    sig = (probs_all > best_thr) & down_mask
    n_trades = sig.sum()
    if n_trades > 0:
        prec = np.mean(y_all[sig] == 1)
        # Assume: win = +0.5% (reversal happens), lose = -0.3% (stop loss)
        win_pct, loss_pct = 0.5, 0.3
        avg_return = prec * win_pct - (1 - prec) * loss_pct
        total_return = avg_return * n_trades
        print(f"  Trades: {n_trades}")
        print(f"  Win rate: {prec*100:.1f}%")
        print(f"  Avg return per trade: {avg_return*100:.3f}% (win={win_pct}%, loss={loss_pct}%)")
        print(f"  Total return ({n_trades} trades): {total_return:.2f}%")
        print(f"  Annualized (assuming 252 trading days in test): ~{total_return/1:.1f}%")

    print(f"\n  Baseline: Attn-LSTM TP direction = 53.2% (N=2694, p=0.0008)")


run_detailed_eval()

In [ ]:
"""
诊断: 检查所有保存的CNN模型, 找一个能正常输出的
"""
import os
import torch
import torch.nn as nn
import numpy as np

MODEL_DIR = CONFIG['data_dir'] / r'models'

class CNNDualModel(nn.Module):
    def __init__(self, n_features=16, window=60):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=(3, n_features), padding=(1, 0))
        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3, 1), padding=(1, 0))
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc_bottom = nn.Linear(64, 1)
        self.fc_top = nn.Linear(64, 1)
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1).squeeze(-1)
        return self.fc_bottom(x).squeeze(-1), self.fc_top(x).squeeze(-1)

class CNNReversalModel(nn.Module):
    def __init__(self, n_features=16, window=60):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=(3, n_features), padding=(1, 0))
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3, 1), padding=(1, 0))
        self.bn2 = nn.BatchNorm2d(64)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(64, 1)
    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        x = torch.relu(self.bn2(self.conv2(x)))
        x = self.pool(x).squeeze(-1).squeeze(-1)
        x = self.dropout(x)
        return self.fc(x).squeeze(-1)

# 列出所有模型文件
print("All model files:")
for f in sorted(os.listdir(MODEL_DIR)):
    if f.endswith('.pt') or f.endswith('.pth'):
        fp = os.path.join(MODEL_DIR, f)
        size_mb = os.path.getsize(fp) / 1024 / 1024
        print(f"  {f} ({size_mb:.1f} MB)")

print(f"\n{'='*60}")
print("Testing each model with random input...")
print(f"{'='*60}")

# 随机输入
dummy_input = torch.randn(1, 1, 60, 16)

for f in sorted(os.listdir(MODEL_DIR)):
    if not (f.endswith('.pt') or f.endswith('.pth')):
        continue
    fp = os.path.join(MODEL_DIR, f)
    print(f"\n--- {f} ---")
    try:
        ckpt = torch.load(fp, map_location='cpu', weights_only=True)

        if isinstance(ckpt, dict):
            keys = list(ckpt.keys())
            print(f"  Keys: {keys[:10]}")

            # 检查不同的key命名
            state_dict = None
            for k in ['model_state_dict', 'state_dict', 'model_bot_state', 'model_top_state']:
                if k in ckpt:
                    state_dict = ckpt[k]
                    print(f"  Found state_dict in key: '{k}'")
                    break

            if state_dict is None and 'model_state_dict' not in ckpt:
                # 可能ckpt本身就是state_dict
                if any('conv' in k for k in ckpt.keys()):
                    state_dict = ckpt
                    print(f"  Checkpoint IS the state_dict")

            if state_dict:
                # 检查NaN
                has_nan = False
                for k, v in state_dict.items():
                    if torch.isnan(v).any():
                        has_nan = True
                        print(f"  ⚠ NaN in {k}")

                if not has_nan:
                    print(f"  ✓ No NaN weights")

                # 检查权重范围
                all_vals = torch.cat([v.flatten().float() for v in state_dict.values()])
                print(f"  Weight stats: min={all_vals.min():.4f}, max={all_vals.max():.4f}, "
                      f"mean={all_vals.mean():.4f}, std={all_vals.std():.4f}")

                # 试着加载并推理
                try:
                    # 判断是哪种模型
                    if 'fc_bottom.weight' in state_dict:
                        model = CNNDualModel()
                        model.load_state_dict(state_dict)
                        model.eval()
                        with torch.no_grad():
                            pb, pt = model(dummy_input)
                            pb = torch.sigmoid(pb).item()
                            pt = torch.sigmoid(pt).item()
                        print(f"  Output: prob_bot={pb:.4f}, prob_top={pt:.4f}")
                        if pb > 0.01 and pb < 0.99:
                            print(f"  ✓ MODEL WORKS!")
                    elif 'fc.weight' in state_dict:
                        model = CNNReversalModel()
                        model.load_state_dict(state_dict)
                        model.eval()
                        with torch.no_grad():
                            out = model(dummy_input)
                            prob = torch.sigmoid(out).item()
                        print(f"  Output: prob={prob:.4f}")
                        if prob > 0.01 and prob < 0.99:
                            print(f"  ✓ MODEL WORKS!")
                    else:
                        print(f"  Unknown model architecture, keys: {list(state_dict.keys())[:5]}")
                except Exception as e:
                    print(f"  Load error: {e}")

            # 检查scaler
            if 'scaler_mean' in ckpt:
                sm = ckpt['scaler_mean']
                ss = ckpt['scaler_scale']
                print(f"  Scaler: mean shape={sm.shape}, any NaN={np.any(np.isnan(sm))}")
        else:
            print(f"  Not a dict, type={type(ckpt)}")
    except Exception as e:
        print(f"  Error loading: {e}")

In [ ]:
"""
CNN 重训: 修复标签 + 5分钟bar
=====================================
三种标签定义对比:
  A. "end_point": close[t+LA] > close[t]  (未来LA bar后价格确实更高)
  B. "sustained": mean(close[t+LA//2:t+LA]) > close[t]  (后半段均价更高)
  C. "strong":    close[t+LA]/close[t] - 1 > rev_pct  (反弹幅度超过阈值)

5分钟bar: 时间跨度不变，bar数×3
  TREND_LOOKBACK: 6×15min=90min → 18×5min=90min
  CNN_LOOKAHEAD:  6×15min=90min → 18×5min=90min
  CNN_WINDOW:     30×15min=7.5h → 60×5min=5h (稍短,训练更快)
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIG
# ============================================================
DRIVE_BASE = CONFIG['data_dir'] / r'splits'
SAVE_DIR = CONFIG['data_dir'] / r'models'
FREQ_RAW = "1min"
RESAMPLE_PERIOD = 5  # ← 改为5分钟
TICKERS = ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
TRAIN_RATIO = 0.8

# CNN config (时间跨度和15min版本一致)
CNN_WINDOW = 60          # 60×5min = 5小时 (15min版: 30×15min=7.5h)
CNN_N_FEATURES = 16
CNN_EPOCHS = 80
CNN_PATIENCE = 15
CNN_BATCH = 64
CNN_LR = 1e-3

# 趋势/反转参数 (时间跨度不变)
TREND_PCT = 0.5          # 0.5% 趋势阈值
TREND_LOOKBACK = 18      # 18×5min = 90min (和15min×6一样)
CNN_LOOKAHEAD = 18       # 18×5min = 90min

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")
print(f"Resample: {RESAMPLE_PERIOD}min")
print(f"CNN Window: {CNN_WINDOW} bars ({CNN_WINDOW*RESAMPLE_PERIOD}min)")
print(f"Trend Lookback: {TREND_LOOKBACK} bars ({TREND_LOOKBACK*RESAMPLE_PERIOD}min)")
print(f"CNN Lookahead: {CNN_LOOKAHEAD} bars ({CNN_LOOKAHEAD*RESAMPLE_PERIOD}min)")
os.makedirs(SAVE_DIR, exist_ok=True)


# ============================================================
# DATA LOADING
# ============================================================
def load_split_csv(csv_path):
    df = pd.read_csv(csv_path)
    col_map = {}
    for col in df.columns:
        cl = col.lower().strip()
        if cl == 'ts_event': col_map[col] = 'timestamp'
        elif cl == 'open': col_map[col] = 'open'
        elif cl == 'high': col_map[col] = 'high'
        elif cl == 'low': col_map[col] = 'low'
        elif cl == 'close': col_map[col] = 'close'
        elif cl == 'volume': col_map[col] = 'volume'
    df = df.rename(columns=col_map)
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df

def resample(df, period):
    df = df.set_index('timestamp').sort_index()
    resampled = df.resample(f'{period}min').agg({
        'open': 'first', 'high': 'max', 'low': 'min',
        'close': 'last', 'volume': 'sum'
    }).dropna(subset=['close'])
    return resampled.reset_index()

def load_ticker(ticker):
    data_dir = os.path.join(DRIVE_BASE, f"{ticker}_{FREQ_RAW}")
    dfs = []
    for split in ['train', 'test']:
        fpath = os.path.join(data_dir, f"{split}.csv")
        if os.path.exists(fpath):
            dfs.append(load_split_csv(fpath))
    if not dfs:
        print(f"  [SKIP] {ticker}")
        return None
    df = pd.concat(dfs, ignore_index=True)
    df = df.sort_values('timestamp').reset_index(drop=True)
    df = resample(df, RESAMPLE_PERIOD)
    return df


# ============================================================
# MODEL
# ============================================================
class CNNDualModel(nn.Module):
    def __init__(self, n_features=16, window=60):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=(3, n_features), padding=(1, 0))
        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3, 1), padding=(1, 0))
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc_bottom = nn.Linear(64, 1)
        self.fc_top = nn.Linear(64, 1)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1).squeeze(-1)
        return self.fc_bottom(x).squeeze(-1), self.fc_top(x).squeeze(-1)


# ============================================================
# 16 FEATURES (和原版一致)
# ============================================================
def compute_cnn_features_16(df):
    close = df['close'].values.astype(float)
    high = df['high'].values.astype(float)
    low = df['low'].values.astype(float)
    volume = df['volume'].values.astype(float)

    feat = pd.DataFrame()
    ret = pd.Series(close).pct_change()

    feat['ret_1'] = ret.values
    feat['ret_5'] = ret.rolling(5).sum().values
    feat['ret_15'] = ret.rolling(15).sum().values
    feat['high_low_range'] = (high - low) / (close + 1e-10)
    feat['close_position'] = (close - low) / (high - low + 1e-10)

    sma5 = pd.Series(close).rolling(5).mean()
    sma15 = pd.Series(close).rolling(15).mean()
    sma30 = pd.Series(close).rolling(30).mean()
    feat['ma5_15'] = ((sma5 - sma15) / (sma15 + 1e-10)).values
    feat['ma5_30'] = ((sma5 - sma30) / (sma30 + 1e-10)).values

    feat['vol_5'] = ret.rolling(5).std().values
    feat['vol_15'] = ret.rolling(15).std().values

    delta = ret.copy()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss_v = (-delta).clip(lower=0).rolling(14).mean()
    feat['rsi'] = (100 - 100 / (1 + gain / (loss_v + 1e-10))).values

    feat['vol_ratio'] = volume / (pd.Series(volume).rolling(20).mean().values + 1e-10)

    sma20 = pd.Series(close).rolling(20).mean()
    std20 = pd.Series(close).rolling(20).std()
    feat['bb_pos'] = ((close - sma20) / (2 * std20 + 1e-10)).values

    ema12 = pd.Series(close).ewm(span=12).mean()
    ema26 = pd.Series(close).ewm(span=26).mean()
    feat['macd'] = ((ema12 - ema26) / (close + 1e-10)).values

    tr = np.maximum(high - low, np.maximum(
        np.abs(high - np.roll(close, 1)), np.abs(low - np.roll(close, 1))))
    feat['atr'] = (pd.Series(tr).rolling(14).mean() / (close + 1e-10)).values

    cols = list(feat.columns)[:CNN_N_FEATURES]
    while len(cols) < CNN_N_FEATURES:
        cols.append(cols[-1])
    feat = feat[cols]
    return feat.values.astype(np.float32)


# ============================================================
# 三种标签定义
# ============================================================
def find_trend_and_label(df, label_type='end_point'):
    """
    标签类型:
      'original':   未来LA bar内最高价弹>rev_pct (原版,包含假反弹)
      'end_point':  未来第LA bar价格 > 当前价格 (真反转)
      'sustained':  未来后半段均价 > 当前价格 (持续反转)
      'strong':     未来第LA bar价格比当前高>rev_pct (强反转)
    """
    close = df['close'].values.astype(float)
    n = len(close)
    trend_pct = TREND_PCT / 100.0
    LB = TREND_LOOKBACK
    LA = CNN_LOOKAHEAD

    positions = []
    for t in range(max(CNN_WINDOW, LB), n - LA):
        p_now = close[t]
        p_past = close[t - LB]
        if p_now <= 0 or p_past <= 0:
            continue
        move = (p_now - p_past) / p_past

        if move < -trend_pct:
            # 下跌趋势 → 检测底部反转
            if label_type == 'original':
                future_max = np.max(close[t+1:t+1+LA])
                label = 1 if (future_max - p_now) / p_now > trend_pct else 0
            elif label_type == 'end_point':
                label = 1 if close[t + LA] > p_now else 0
            elif label_type == 'sustained':
                half = LA // 2
                future_mean = np.mean(close[t + half:t + LA])
                label = 1 if future_mean > p_now else 0
            elif label_type == 'strong':
                label = 1 if (close[t + LA] - p_now) / p_now > trend_pct else 0
            positions.append({'bar_idx': t, 'trend_dir': 'down', 'label': label})

        elif move > trend_pct:
            # 上涨趋势 → 检测顶部反转
            if label_type == 'original':
                future_min = np.min(close[t+1:t+1+LA])
                label = 1 if (p_now - future_min) / p_now > trend_pct else 0
            elif label_type == 'end_point':
                label = 1 if close[t + LA] < p_now else 0
            elif label_type == 'sustained':
                half = LA // 2
                future_mean = np.mean(close[t + half:t + LA])
                label = 1 if future_mean < p_now else 0
            elif label_type == 'strong':
                label = 1 if (p_now - close[t + LA]) / p_now > trend_pct else 0
            positions.append({'bar_idx': t, 'trend_dir': 'up', 'label': label})

    return positions


def build_cnn_dataset(features_16, positions):
    X, y = [], []
    for p in positions:
        idx = p['bar_idx']
        if idx < CNN_WINDOW:
            continue
        window = features_16[idx - CNN_WINDOW:idx]
        if window.shape != (CNN_WINDOW, CNN_N_FEATURES):
            continue
        X.append(window)
        y.append(p['label'])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)


# ============================================================
# 训练一个CNN (给定标签类型)
# ============================================================
def train_cnn_for_label_type(all_ticker_data, label_type):
    print(f"\n{'='*60}")
    print(f"  Training CNN — label_type='{label_type}'")
    print(f"{'='*60}")

    down_tr_X, down_tr_y, down_v_X, down_v_y = [], [], [], []
    up_tr_X, up_tr_y, up_v_X, up_v_y = [], [], [], []

    for ticker, (df, features_16, train_end_idx) in all_ticker_data.items():
        positions = find_trend_and_label(df, label_type=label_type)

        pos_down = [p for p in positions if p['bar_idx'] < train_end_idx and p['trend_dir'] == 'down']
        pos_up = [p for p in positions if p['bar_idx'] < train_end_idx and p['trend_dir'] == 'up']

        if len(pos_down) > 30:
            X_d, y_d = build_cnn_dataset(features_16, pos_down)
            if len(X_d) > 10:
                vs = max(int(len(X_d) * 0.15), 1)
                down_tr_X.append(X_d[:-vs]); down_tr_y.append(y_d[:-vs])
                down_v_X.append(X_d[-vs:]); down_v_y.append(y_d[-vs:])

        if len(pos_up) > 30:
            X_u, y_u = build_cnn_dataset(features_16, pos_up)
            if len(X_u) > 10:
                vs = max(int(len(X_u) * 0.15), 1)
                up_tr_X.append(X_u[:-vs]); up_tr_y.append(y_u[:-vs])
                up_v_X.append(X_u[-vs:]); up_v_y.append(y_u[-vs:])

    if not down_tr_X or not up_tr_X:
        print("  [ERROR] Insufficient data")
        return None, None, None

    Xd_tr = np.concatenate(down_tr_X); yd_tr = np.concatenate(down_tr_y)
    Xd_v = np.concatenate(down_v_X); yd_v = np.concatenate(down_v_y)
    Xu_tr = np.concatenate(up_tr_X); yu_tr = np.concatenate(up_tr_y)
    Xu_v = np.concatenate(up_v_X); yu_v = np.concatenate(up_v_y)

    pos_rate_d = yd_tr.mean()
    pos_rate_u = yu_tr.mean()
    print(f"  Bottom: train={len(Xd_tr)}, val={len(Xd_v)}, pos_rate={pos_rate_d:.3f}")
    print(f"  Top:    train={len(Xu_tr)}, val={len(Xu_v)}, pos_rate={pos_rate_u:.3f}")

    # Single scaler
    all_train_X = np.concatenate([Xd_tr, Xu_tr], axis=0)
    n_total, W, F = all_train_X.shape
    flat_all = all_train_X.reshape(-1, F)
    sc = StandardScaler().fit(flat_all)
    scaler_mean = sc.mean_.astype(np.float32)
    scaler_scale = sc.scale_.astype(np.float32)

    Xd_tr = sc.transform(Xd_tr.reshape(-1, F)).reshape(len(Xd_tr), W, F)
    Xd_v = sc.transform(Xd_v.reshape(-1, F)).reshape(len(Xd_v), W, F)
    Xu_tr = sc.transform(Xu_tr.reshape(-1, F)).reshape(len(Xu_tr), W, F)
    Xu_v = sc.transform(Xu_v.reshape(-1, F)).reshape(len(Xu_v), W, F)

    for arr in [Xd_tr, Xd_v, Xu_tr, Xu_v]:
        arr[np.isnan(arr)] = 0

    # Train
    model = CNNDualModel(n_features=CNN_N_FEATURES, window=CNN_WINDOW).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=CNN_LR, weight_decay=1e-5)
    crit = nn.BCEWithLogitsLoss()

    ds_d = TensorDataset(torch.FloatTensor(Xd_tr).unsqueeze(1), torch.FloatTensor(yd_tr.astype(np.float32)))
    dl_d = DataLoader(ds_d, batch_size=CNN_BATCH, shuffle=True)
    ds_u = TensorDataset(torch.FloatTensor(Xu_tr).unsqueeze(1), torch.FloatTensor(yu_tr.astype(np.float32)))
    dl_u = DataLoader(ds_u, batch_size=CNN_BATCH, shuffle=True)

    Xdv_t = torch.FloatTensor(Xd_v).unsqueeze(1).to(DEVICE)
    ydv_t = torch.FloatTensor(yd_v.astype(np.float32)).to(DEVICE)
    Xuv_t = torch.FloatTensor(Xu_v).unsqueeze(1).to(DEVICE)
    yuv_t = torch.FloatTensor(yu_v.astype(np.float32)).to(DEVICE)

    best_vl, best_st, pat = float('inf'), None, 0
    for ep in range(CNN_EPOCHS):
        model.train()
        for xb, yb in dl_d:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss = crit(model(xb)[0], yb); loss.backward(); opt.step()
        for xb, yb in dl_u:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss = crit(model(xb)[1], yb); loss.backward(); opt.step()

        model.eval()
        with torch.no_grad():
            vl_b = crit(model(Xdv_t)[0], ydv_t).item()
            vl_t = crit(model(Xuv_t)[1], yuv_t).item()
            vl = (vl_b + vl_t) / 2

        if vl < best_vl:
            best_vl, best_st, pat = vl, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            pat += 1
            if pat >= CNN_PATIENCE:
                print(f"  Early stop epoch {ep+1}")
                break
        if (ep + 1) % 20 == 0:
            print(f"  Epoch {ep+1}: val_loss={vl:.4f}")

    if best_st:
        model.load_state_dict(best_st)

    model = model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        pb = torch.sigmoid(model(Xdv_t)[0]).cpu().numpy()
        pt = torch.sigmoid(model(Xuv_t)[1]).cpu().numpy()
    acc_b = np.mean((pb > 0.5).astype(int) == yd_v) * 100
    acc_t = np.mean((pt > 0.5).astype(int) == yu_v) * 100
    print(f"  Val Accuracy — Bottom: {acc_b:.1f}%, Top: {acc_t:.1f}%")

    # Save
    save_dict = {
        'model_state_dict': model.state_dict(),
        'scaler_mean': scaler_mean,
        'scaler_scale': scaler_scale,
        'n_features': CNN_N_FEATURES,
        'window': CNN_WINDOW,
        'label_type': label_type,
        'resample_period': RESAMPLE_PERIOD,
    }
    save_path = os.path.join(SAVE_DIR, f"cnn_dual_5min_{label_type}.pt")
    torch.save(save_dict, save_path)
    print(f"  ✓ Saved to {save_path}")

    return model, scaler_mean, scaler_scale


# ============================================================
# INFERENCE + DIAGNOSTIC
# ============================================================
@torch.no_grad()
def run_cnn_inference(df, model, scaler_mean, scaler_scale, device='cpu'):
    features_16 = compute_cnn_features_16(df)
    features_normed = (features_16 - scaler_mean) / (scaler_scale + 1e-8)
    features_normed = np.nan_to_num(features_normed, nan=0.0, posinf=0.0, neginf=0.0)

    prob_bottom = np.full(len(df), np.nan)
    prob_top = np.full(len(df), np.nan)

    batch_size = 512
    for start in range(CNN_WINDOW, len(features_normed), batch_size):
        end = min(start + batch_size, len(features_normed))
        batch_w, batch_idx = [], []
        for i in range(start, end):
            w = features_normed[i - CNN_WINDOW:i]
            if w.shape == (CNN_WINDOW, CNN_N_FEATURES):
                batch_w.append(w); batch_idx.append(i)
        if not batch_w:
            continue
        x = torch.FloatTensor(np.array(batch_w)).unsqueeze(1).to(device)
        pb, pt = model(x)
        for j, idx in enumerate(batch_idx):
            prob_bottom[idx] = torch.sigmoid(pb[j]).item()
            prob_top[idx] = torch.sigmoid(pt[j]).item()

    result = pd.DataFrame(index=df.index)
    result['cnn_prob_bottom'] = np.nan_to_num(prob_bottom, nan=0.5)
    result['cnn_prob_top'] = np.nan_to_num(prob_top, nan=0.5)
    return result


def evaluate_direction(stock, df, cnn_features, label_type):
    """在趋势bar上测试CNN能否预测真方向"""
    close = df['close'].values
    n = len(close)
    trend_pct = TREND_PCT / 100.0
    LA = CNN_LOOKAHEAD

    print(f"\n  {stock} [{label_type}]: Direction Test")
    print(f"  {'Type':<12} {'Thr':>5} {'N':>6} {'DA':>8} {'p':>10} {'Sig':>5} {'AvgRet':>10}")

    for horizon in [LA]:  # 用和标签一致的horizon
        future_ret = pd.Series(close).pct_change(horizon).shift(-horizon)

        for thr in [0.3, 0.4, 0.5, 0.6, 0.7]:
            # 下跌趋势 + CNN bottom → 价格涨了吗?
            downtrend = np.zeros(n, dtype=bool)
            uptrend = np.zeros(n, dtype=bool)
            for t in range(TREND_LOOKBACK, n):
                move = (close[t] - close[t - TREND_LOOKBACK]) / (close[t - TREND_LOOKBACK] + 1e-10)
                if move < -trend_pct:
                    downtrend[t] = True
                elif move > trend_pct:
                    uptrend[t] = True

            mask = downtrend & (cnn_features['cnn_prob_bottom'] > thr)
            rets = future_ret[mask].dropna()
            if len(rets) >= 10:
                n_up = int((rets > 0).sum())
                da = n_up / len(rets)
                p_val = stats.binomtest(n_up, len(rets), 0.5, alternative='greater').pvalue
                sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
                print(f"  {'DT+Bottom':<12} {thr:>5.1f} {len(rets):>6} {da:>8.4f} {p_val:>10.4f} {sig:>5} {rets.mean():>10.6f}")

            mask = uptrend & (cnn_features['cnn_prob_top'] > thr)
            rets = future_ret[mask].dropna()
            if len(rets) >= 10:
                n_down = int((rets < 0).sum())
                da = n_down / len(rets)
                p_val = stats.binomtest(n_down, len(rets), 0.5, alternative='greater').pvalue
                sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
                print(f"  {'UT+Top':<12} {thr:>5.1f} {len(rets):>6} {da:>8.4f} {p_val:>10.4f} {sig:>5} {rets.mean():>10.6f}")

        # 基线对比
        mask_all_dt = downtrend & pd.notna(future_ret)
        rets_base = future_ret[mask_all_dt].dropna()
        if len(rets_base) >= 10:
            base_da = (rets_base > 0).mean()
            print(f"  {'Baseline DT':<12} {'all':>5} {len(rets_base):>6} {base_da:>8.4f} {'':>10} {'':>5} {rets_base.mean():>10.6f}")


# ============================================================
# MAIN: 加载数据 → 训练3种标签 → 诊断
# ============================================================
print(f"\n{'#'*60}")
print(f"# LOADING 5-MIN DATA")
print(f"{'#'*60}")

cnn_ticker_data = {}
for ticker in TICKERS:
    df = load_ticker(ticker)
    if df is None:
        continue
    n = len(df)
    train_end_idx = int(n * TRAIN_RATIO)
    features_16 = compute_cnn_features_16(df)
    cnn_ticker_data[ticker] = (df, features_16, train_end_idx)
    print(f"  {ticker}: {n} bars, train_end={train_end_idx}")


# 训练3种标签
label_types = ['end_point', 'sustained', 'strong']
models = {}

for lt in label_types:
    model, sc_mean, sc_scale = train_cnn_for_label_type(cnn_ticker_data, lt)
    if model is not None:
        models[lt] = (model, sc_mean, sc_scale)


# ============================================================
# DIAGNOSTIC: 每种标签 × 每只股票
# ============================================================
print(f"\n\n{'#'*60}")
print(f"# DIRECTION DIAGNOSTIC (5-min bars)")
print(f"{'#'*60}")

test_stocks = ['AAPL', 'MSFT', 'SPY']

for stock in test_stocks:
    if stock not in cnn_ticker_data:
        continue

    df_full, _, _ = cnn_ticker_data[stock]
    n = len(df_full)
    test_start = int(n * TRAIN_RATIO)
    df_test = df_full.iloc[test_start:].reset_index(drop=True)
    print(f"\n\n{'='*60}")
    print(f"  {stock}: {len(df_test)} test bars")
    print(f"{'='*60}")

    for lt, (model, sc_mean, sc_scale) in models.items():
        cnn_feat = run_cnn_inference(df_test, model, sc_mean, sc_scale, str(DEVICE))
        evaluate_direction(stock, df_test, cnn_feat, lt)


# ============================================================
# SUMMARY
# ============================================================
print(f"\n\n{'#'*60}")
print(f"# SUMMARY")
print(f"{'#'*60}")
print(f"""
标签定义:
  end_point: close[t+{CNN_LOOKAHEAD}] > close[t]
             → 未来{CNN_LOOKAHEAD*RESAMPLE_PERIOD}min后价格确实更高
             → 如果DA显著>50%, CNN能预测真反转

  sustained: mean(close[后半段]) > close[t]
             → 未来均价更高(不是摸一下)
             → 排除V型假反弹

  strong:    (close[t+{CNN_LOOKAHEAD}] - close[t]) / close[t] > {TREND_PCT}%
             → 反弹幅度必须>阈值
             → 最严格的定义

如果某种标签DA显著>50%, 用那种标签重训最终CNN
如果全部≈50%, 说明价格形态无法预测未来方向(EMH)
""")

In [ ]:
"""
CNN 迁移学习: strong预训练 → 方向微调
==========================================
Step 1: 加载已训练好的strong模型 (conv层学会了反转形态, 81%/85%)
Step 2: 冻结conv层, 只微调fc层去预测真方向 (end_point/sustained)
Step 3: 对比直接训练 vs 迁移学习的效果

同时测试几种微调策略:
  A. 只微调fc (最保守)
  B. 微调conv2+fc (中等)
  C. 全部微调但低学习率 (最激进)
  D. 加dropout+新fc层 (更强的方向头)
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIG (和5min训练一致)
# ============================================================
DRIVE_BASE = CONFIG['data_dir'] / r'splits'
SAVE_DIR = CONFIG['data_dir'] / r'models'
FREQ_RAW = "1min"
RESAMPLE_PERIOD = 5
TICKERS = ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
TRAIN_RATIO = 0.8

CNN_WINDOW = 60
CNN_N_FEATURES = 16
CNN_BATCH = 64
TREND_PCT = 0.5
TREND_LOOKBACK = 18
CNN_LOOKAHEAD = 18

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")


# ============================================================
# DATA LOADING (复用)
# ============================================================
def load_split_csv(csv_path):
    df = pd.read_csv(csv_path)
    col_map = {}
    for col in df.columns:
        cl = col.lower().strip()
        if cl == 'ts_event': col_map[col] = 'timestamp'
        elif cl == 'open': col_map[col] = 'open'
        elif cl == 'high': col_map[col] = 'high'
        elif cl == 'low': col_map[col] = 'low'
        elif cl == 'close': col_map[col] = 'close'
        elif cl == 'volume': col_map[col] = 'volume'
    df = df.rename(columns=col_map)
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df

def resample(df, period):
    df = df.set_index('timestamp').sort_index()
    resampled = df.resample(f'{period}min').agg({
        'open': 'first', 'high': 'max', 'low': 'min',
        'close': 'last', 'volume': 'sum'
    }).dropna(subset=['close'])
    return resampled.reset_index()

def load_ticker(ticker):
    data_dir = os.path.join(DRIVE_BASE, f"{ticker}_{FREQ_RAW}")
    dfs = []
    for split in ['train', 'test']:
        fpath = os.path.join(data_dir, f"{split}.csv")
        if os.path.exists(fpath):
            dfs.append(load_split_csv(fpath))
    if not dfs:
        return None
    df = pd.concat(dfs, ignore_index=True)
    df = df.sort_values('timestamp').reset_index(drop=True)
    df = resample(df, RESAMPLE_PERIOD)
    return df

def compute_cnn_features_16(df):
    close = df['close'].values.astype(float)
    high = df['high'].values.astype(float)
    low = df['low'].values.astype(float)
    volume = df['volume'].values.astype(float)
    feat = pd.DataFrame()
    ret = pd.Series(close).pct_change()
    feat['ret_1'] = ret.values
    feat['ret_5'] = ret.rolling(5).sum().values
    feat['ret_15'] = ret.rolling(15).sum().values
    feat['high_low_range'] = (high - low) / (close + 1e-10)
    feat['close_position'] = (close - low) / (high - low + 1e-10)
    sma5 = pd.Series(close).rolling(5).mean()
    sma15 = pd.Series(close).rolling(15).mean()
    sma30 = pd.Series(close).rolling(30).mean()
    feat['ma5_15'] = ((sma5 - sma15) / (sma15 + 1e-10)).values
    feat['ma5_30'] = ((sma5 - sma30) / (sma30 + 1e-10)).values
    feat['vol_5'] = ret.rolling(5).std().values
    feat['vol_15'] = ret.rolling(15).std().values
    delta = ret.copy()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss_v = (-delta).clip(lower=0).rolling(14).mean()
    feat['rsi'] = (100 - 100 / (1 + gain / (loss_v + 1e-10))).values
    feat['vol_ratio'] = volume / (pd.Series(volume).rolling(20).mean().values + 1e-10)
    sma20 = pd.Series(close).rolling(20).mean()
    std20 = pd.Series(close).rolling(20).std()
    feat['bb_pos'] = ((close - sma20) / (2 * std20 + 1e-10)).values
    ema12 = pd.Series(close).ewm(span=12).mean()
    ema26 = pd.Series(close).ewm(span=26).mean()
    feat['macd'] = ((ema12 - ema26) / (close + 1e-10)).values
    tr = np.maximum(high - low, np.maximum(
        np.abs(high - np.roll(close, 1)), np.abs(low - np.roll(close, 1))))
    feat['atr'] = (pd.Series(tr).rolling(14).mean() / (close + 1e-10)).values
    cols = list(feat.columns)[:CNN_N_FEATURES]
    while len(cols) < CNN_N_FEATURES:
        cols.append(cols[-1])
    feat = feat[cols]
    return feat.values.astype(np.float32)


# ============================================================
# MODELS
# ============================================================
class CNNDualModel(nn.Module):
    """原始CNN (和strong预训练一致)"""
    def __init__(self, n_features=16, window=60):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=(3, n_features), padding=(1, 0))
        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3, 1), padding=(1, 0))
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc_bottom = nn.Linear(64, 1)
        self.fc_top = nn.Linear(64, 1)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1).squeeze(-1)
        return self.fc_bottom(x).squeeze(-1), self.fc_top(x).squeeze(-1)


class CNNTransferModel(nn.Module):
    """迁移学习版: conv层来自预训练, 新的direction head更强"""
    def __init__(self, n_features=16, window=60):
        super().__init__()
        # 这些从预训练模型加载
        self.conv1 = nn.Conv2d(1, 32, kernel_size=(3, n_features), padding=(1, 0))
        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3, 1), padding=(1, 0))
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        # 新的direction head (更强, 带dropout)
        self.dir_bottom = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )
        self.dir_top = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1).squeeze(-1)
        return self.dir_bottom(x).squeeze(-1), self.dir_top(x).squeeze(-1)

    def load_pretrained_conv(self, state_dict):
        """只加载conv层权重"""
        self.conv1.load_state_dict({
            k.replace('conv1.', ''): v for k, v in state_dict.items() if k.startswith('conv1.')
        })
        self.conv2.load_state_dict({
            k.replace('conv2.', ''): v for k, v in state_dict.items() if k.startswith('conv2.')
        })
        # pool没有参数, fc不加载(用新的direction head)


# ============================================================
# LABELS
# ============================================================
def find_trend_and_label(df, label_type='sustained'):
    close = df['close'].values.astype(float)
    n = len(close)
    trend_pct = TREND_PCT / 100.0
    LB = TREND_LOOKBACK
    LA = CNN_LOOKAHEAD
    positions = []
    for t in range(max(CNN_WINDOW, LB), n - LA):
        p_now = close[t]
        p_past = close[t - LB]
        if p_now <= 0 or p_past <= 0:
            continue
        move = (p_now - p_past) / p_past
        if move < -trend_pct:
            if label_type == 'end_point':
                label = 1 if close[t + LA] > p_now else 0
            elif label_type == 'sustained':
                half = LA // 2
                label = 1 if np.mean(close[t + half:t + LA]) > p_now else 0
            elif label_type == 'strong':
                label = 1 if (close[t + LA] - p_now) / p_now > trend_pct else 0
            positions.append({'bar_idx': t, 'trend_dir': 'down', 'label': label})
        elif move > trend_pct:
            if label_type == 'end_point':
                label = 1 if close[t + LA] < p_now else 0
            elif label_type == 'sustained':
                half = LA // 2
                label = 1 if np.mean(close[t + half:t + LA]) < p_now else 0
            elif label_type == 'strong':
                label = 1 if (p_now - close[t + LA]) / p_now > trend_pct else 0
            positions.append({'bar_idx': t, 'trend_dir': 'up', 'label': label})
    return positions

def build_cnn_dataset(features_16, positions):
    X, y = [], []
    for p in positions:
        idx = p['bar_idx']
        if idx < CNN_WINDOW:
            continue
        window = features_16[idx - CNN_WINDOW:idx]
        if window.shape != (CNN_WINDOW, CNN_N_FEATURES):
            continue
        X.append(window)
        y.append(p['label'])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)


# ============================================================
# BUILD DATASETS
# ============================================================
def build_all_datasets(all_ticker_data, label_type):
    """从所有ticker构建训练/验证集"""
    down_tr_X, down_tr_y, down_v_X, down_v_y = [], [], [], []
    up_tr_X, up_tr_y, up_v_X, up_v_y = [], [], [], []

    for ticker, (df, features_16, train_end_idx) in all_ticker_data.items():
        positions = find_trend_and_label(df, label_type=label_type)
        pos_down = [p for p in positions if p['bar_idx'] < train_end_idx and p['trend_dir'] == 'down']
        pos_up = [p for p in positions if p['bar_idx'] < train_end_idx and p['trend_dir'] == 'up']

        if len(pos_down) > 30:
            X_d, y_d = build_cnn_dataset(features_16, pos_down)
            if len(X_d) > 10:
                vs = max(int(len(X_d) * 0.15), 1)
                down_tr_X.append(X_d[:-vs]); down_tr_y.append(y_d[:-vs])
                down_v_X.append(X_d[-vs:]); down_v_y.append(y_d[-vs:])
        if len(pos_up) > 30:
            X_u, y_u = build_cnn_dataset(features_16, pos_up)
            if len(X_u) > 10:
                vs = max(int(len(X_u) * 0.15), 1)
                up_tr_X.append(X_u[:-vs]); up_tr_y.append(y_u[:-vs])
                up_v_X.append(X_u[-vs:]); up_v_y.append(y_u[-vs:])

    Xd_tr = np.concatenate(down_tr_X); yd_tr = np.concatenate(down_tr_y)
    Xd_v = np.concatenate(down_v_X); yd_v = np.concatenate(down_v_y)
    Xu_tr = np.concatenate(up_tr_X); yu_tr = np.concatenate(up_tr_y)
    Xu_v = np.concatenate(up_v_X); yu_v = np.concatenate(up_v_y)

    # Scaler (用strong模型的scaler保持一致)
    all_train_X = np.concatenate([Xd_tr, Xu_tr], axis=0)
    n_total, W, F = all_train_X.shape
    sc = StandardScaler().fit(all_train_X.reshape(-1, F))
    sc_mean = sc.mean_.astype(np.float32)
    sc_scale = sc.scale_.astype(np.float32)

    Xd_tr = sc.transform(Xd_tr.reshape(-1, F)).reshape(len(Xd_tr), W, F)
    Xd_v = sc.transform(Xd_v.reshape(-1, F)).reshape(len(Xd_v), W, F)
    Xu_tr = sc.transform(Xu_tr.reshape(-1, F)).reshape(len(Xu_tr), W, F)
    Xu_v = sc.transform(Xu_v.reshape(-1, F)).reshape(len(Xu_v), W, F)

    for arr in [Xd_tr, Xd_v, Xu_tr, Xu_v]:
        arr[np.isnan(arr)] = 0

    return (Xd_tr, yd_tr, Xd_v, yd_v, Xu_tr, yu_tr, Xu_v, yu_v, sc_mean, sc_scale)


# ============================================================
# TRAINING STRATEGIES
# ============================================================
def train_strategy(model, datasets, strategy_name, lr=1e-3, epochs=60, patience=12):
    """通用训练循环"""
    Xd_tr, yd_tr, Xd_v, yd_v, Xu_tr, yu_tr, Xu_v, yu_v = datasets

    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                           lr=lr, weight_decay=1e-5)
    crit = nn.BCEWithLogitsLoss()

    ds_d = TensorDataset(torch.FloatTensor(Xd_tr).unsqueeze(1), torch.FloatTensor(yd_tr.astype(np.float32)))
    dl_d = DataLoader(ds_d, batch_size=CNN_BATCH, shuffle=True)
    ds_u = TensorDataset(torch.FloatTensor(Xu_tr).unsqueeze(1), torch.FloatTensor(yu_tr.astype(np.float32)))
    dl_u = DataLoader(ds_u, batch_size=CNN_BATCH, shuffle=True)

    Xdv_t = torch.FloatTensor(Xd_v).unsqueeze(1).to(DEVICE)
    ydv_t = torch.FloatTensor(yd_v.astype(np.float32)).to(DEVICE)
    Xuv_t = torch.FloatTensor(Xu_v).unsqueeze(1).to(DEVICE)
    yuv_t = torch.FloatTensor(yu_v.astype(np.float32)).to(DEVICE)

    best_vl, best_st, pat = float('inf'), None, 0
    for ep in range(epochs):
        model.train()
        for xb, yb in dl_d:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss = crit(model(xb)[0], yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        for xb, yb in dl_u:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss = crit(model(xb)[1], yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        with torch.no_grad():
            vl_b = crit(model(Xdv_t)[0], ydv_t).item()
            vl_t = crit(model(Xuv_t)[1], yuv_t).item()
            vl = (vl_b + vl_t) / 2

        if vl < best_vl:
            best_vl, best_st, pat = vl, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            pat += 1
            if pat >= patience:
                break
        if (ep + 1) % 15 == 0:
            print(f"    [{strategy_name}] Epoch {ep+1}: val_loss={vl:.4f}")

    if best_st:
        model.load_state_dict(best_st)

    model = model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        pb = torch.sigmoid(model(Xdv_t)[0]).cpu().numpy()
        pt = torch.sigmoid(model(Xuv_t)[1]).cpu().numpy()
    acc_b = np.mean((pb > 0.5).astype(int) == yd_v) * 100
    acc_t = np.mean((pt > 0.5).astype(int) == yu_v) * 100

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"  [{strategy_name}] Val Acc: Bottom={acc_b:.1f}%, Top={acc_t:.1f}% "
          f"| trainable={n_trainable}, frozen={n_frozen} | early_stop={ep+1-patience if pat>=patience else ep+1}")

    return model, acc_b, acc_t


# ============================================================
# INFERENCE + DIAGNOSTIC
# ============================================================
@torch.no_grad()
def run_inference(df, model, scaler_mean, scaler_scale):
    features_16 = compute_cnn_features_16(df)
    features_normed = (features_16 - scaler_mean) / (scaler_scale + 1e-8)
    features_normed = np.nan_to_num(features_normed, nan=0.0, posinf=0.0, neginf=0.0)

    prob_bottom = np.full(len(df), 0.5)
    prob_top = np.full(len(df), 0.5)

    batch_size = 512
    for start in range(CNN_WINDOW, len(features_normed), batch_size):
        end = min(start + batch_size, len(features_normed))
        batch_w, batch_idx = [], []
        for i in range(start, end):
            w = features_normed[i - CNN_WINDOW:i]
            if w.shape == (CNN_WINDOW, CNN_N_FEATURES):
                batch_w.append(w); batch_idx.append(i)
        if not batch_w:
            continue
        x = torch.FloatTensor(np.array(batch_w)).unsqueeze(1).to(DEVICE)
        pb, pt = model(x)
        for j, idx in enumerate(batch_idx):
            prob_bottom[idx] = torch.sigmoid(pb[j]).item()
            prob_top[idx] = torch.sigmoid(pt[j]).item()

    result = pd.DataFrame(index=df.index)
    result['cnn_prob_bottom'] = prob_bottom
    result['cnn_prob_top'] = prob_top
    return result


def direction_test(stock, df, cnn_features, strategy_name):
    """测试方向预测能力"""
    close = df['close'].values
    n = len(close)
    trend_pct = TREND_PCT / 100.0
    LA = CNN_LOOKAHEAD
    future_ret = pd.Series(close).pct_change(LA).shift(-LA)

    downtrend = np.zeros(n, dtype=bool)
    uptrend = np.zeros(n, dtype=bool)
    for t in range(TREND_LOOKBACK, n):
        move = (close[t] - close[t - TREND_LOOKBACK]) / (close[t - TREND_LOOKBACK] + 1e-10)
        if move < -trend_pct: downtrend[t] = True
        elif move > trend_pct: uptrend[t] = True

    results = []
    for thr in [0.3, 0.4, 0.5, 0.6, 0.7]:
        # Bottom
        mask = downtrend & (cnn_features['cnn_prob_bottom'] > thr)
        rets = future_ret[mask].dropna()
        if len(rets) >= 10:
            n_up = int((rets > 0).sum())
            da = n_up / len(rets)
            p_val = stats.binomtest(n_up, len(rets), 0.5, alternative='greater').pvalue
            results.append(('DT+Bot', thr, len(rets), da, p_val, rets.mean()))

        # Top
        mask = uptrend & (cnn_features['cnn_prob_top'] > thr)
        rets = future_ret[mask].dropna()
        if len(rets) >= 10:
            n_down = int((rets < 0).sum())
            da = n_down / len(rets)
            p_val = stats.binomtest(n_down, len(rets), 0.5, alternative='greater').pvalue
            results.append(('UT+Top', thr, len(rets), da, p_val, rets.mean()))

    # Baseline
    base_rets = future_ret[downtrend].dropna()
    base_da = (base_rets > 0).mean() if len(base_rets) > 0 else 0.5

    return results, base_da, len(base_rets)


# ============================================================
# MAIN
# ============================================================
print(f"\n{'#'*60}")
print(f"# LOADING DATA")
print(f"{'#'*60}")

cnn_ticker_data = {}
for ticker in TICKERS:
    df = load_ticker(ticker)
    if df is None:
        continue
    n = len(df)
    train_end_idx = int(n * TRAIN_RATIO)
    features_16 = compute_cnn_features_16(df)
    cnn_ticker_data[ticker] = (df, features_16, train_end_idx)
    print(f"  {ticker}: {n} bars")


# Load strong pretrained model
strong_path = os.path.join(SAVE_DIR, "cnn_dual_5min_strong.pt")
strong_ckpt = torch.load(strong_path, map_location='cpu', weights_only=True)
strong_state = strong_ckpt['model_state_dict']
print(f"\n  ✓ Loaded strong pretrained model from {strong_path}")


# Build direction datasets (sustained标签, 上轮效果最好)
for label_type in ['sustained', 'end_point']:
    print(f"\n\n{'#'*60}")
    print(f"# LABEL TYPE: {label_type}")
    print(f"{'#'*60}")

    data = build_all_datasets(cnn_ticker_data, label_type)
    Xd_tr, yd_tr, Xd_v, yd_v, Xu_tr, yu_tr, Xu_v, yu_v, sc_mean, sc_scale = data

    print(f"  Bottom: train={len(Xd_tr)}, val={len(Xd_v)}, pos_rate={yd_tr.mean():.3f}")
    print(f"  Top:    train={len(Xu_tr)}, val={len(Xu_v)}, pos_rate={yu_tr.mean():.3f}")

    datasets = (Xd_tr, yd_tr, Xd_v, yd_v, Xu_tr, yu_tr, Xu_v, yu_v)
    all_models = {}

    # --- Strategy 0: Baseline (no pretrain, 直接训练, 作为对比) ---
    print(f"\n  --- Strategy 0: Baseline (no pretrain) ---")
    torch.manual_seed(SEED)
    m0 = CNNDualModel(n_features=CNN_N_FEATURES, window=CNN_WINDOW).to(DEVICE)
    m0, acc0_b, acc0_t = train_strategy(m0, datasets, "baseline", lr=1e-3, epochs=60)
    all_models['baseline'] = (m0, acc0_b, acc0_t)

    # --- Strategy A: 冻结conv, 只微调fc ---
    print(f"\n  --- Strategy A: freeze conv, finetune fc ---")
    torch.manual_seed(SEED)
    mA = CNNDualModel(n_features=CNN_N_FEATURES, window=CNN_WINDOW).to(DEVICE)
    mA.load_state_dict(strong_state)
    mA.conv1.requires_grad_(False)
    mA.conv2.requires_grad_(False)
    # 重新初始化fc层 (不用strong的fc)
    nn.init.xavier_uniform_(mA.fc_bottom.weight)
    nn.init.zeros_(mA.fc_bottom.bias)
    nn.init.xavier_uniform_(mA.fc_top.weight)
    nn.init.zeros_(mA.fc_top.bias)
    mA, accA_b, accA_t = train_strategy(mA, datasets, "freeze_conv", lr=1e-3, epochs=60)
    all_models['freeze_conv'] = (mA, accA_b, accA_t)

    # --- Strategy B: 冻结conv1, 微调conv2+fc ---
    print(f"\n  --- Strategy B: freeze conv1, finetune conv2+fc ---")
    torch.manual_seed(SEED)
    mB = CNNDualModel(n_features=CNN_N_FEATURES, window=CNN_WINDOW).to(DEVICE)
    mB.load_state_dict(strong_state)
    mB.conv1.requires_grad_(False)
    nn.init.xavier_uniform_(mB.fc_bottom.weight)
    nn.init.zeros_(mB.fc_bottom.bias)
    nn.init.xavier_uniform_(mB.fc_top.weight)
    nn.init.zeros_(mB.fc_top.bias)
    mB, accB_b, accB_t = train_strategy(mB, datasets, "freeze_conv1", lr=5e-4, epochs=60)
    all_models['freeze_conv1'] = (mB, accB_b, accB_t)

    # --- Strategy C: 全部微调, 低学习率 ---
    print(f"\n  --- Strategy C: full finetune, low lr ---")
    torch.manual_seed(SEED)
    mC = CNNDualModel(n_features=CNN_N_FEATURES, window=CNN_WINDOW).to(DEVICE)
    mC.load_state_dict(strong_state)
    nn.init.xavier_uniform_(mC.fc_bottom.weight)
    nn.init.zeros_(mC.fc_bottom.bias)
    nn.init.xavier_uniform_(mC.fc_top.weight)
    nn.init.zeros_(mC.fc_top.bias)
    mC, accC_b, accC_t = train_strategy(mC, datasets, "full_finetune", lr=1e-4, epochs=60)
    all_models['full_finetune'] = (mC, accC_b, accC_t)

    # --- Strategy D: 迁移conv + 新的direction head (更强) ---
    print(f"\n  --- Strategy D: transfer conv + new direction head ---")
    torch.manual_seed(SEED)
    mD = CNNTransferModel(n_features=CNN_N_FEATURES, window=CNN_WINDOW).to(DEVICE)
    mD.load_pretrained_conv(strong_state)
    mD.conv1.requires_grad_(False)
    mD.conv2.requires_grad_(False)
    mD, accD_b, accD_t = train_strategy(mD, datasets, "transfer_head", lr=1e-3, epochs=60)
    all_models['transfer_head'] = (mD, accD_b, accD_t)

    # --- Strategy E: 迁移conv + 新head + conv2也微调 ---
    print(f"\n  --- Strategy E: transfer + unfreeze conv2 ---")
    torch.manual_seed(SEED)
    mE = CNNTransferModel(n_features=CNN_N_FEATURES, window=CNN_WINDOW).to(DEVICE)
    mE.load_pretrained_conv(strong_state)
    mE.conv1.requires_grad_(False)  # 只冻结conv1
    mE, accE_b, accE_t = train_strategy(mE, datasets, "transfer_unfreeze", lr=5e-4, epochs=60)
    all_models['transfer_unfreeze'] = (mE, accE_b, accE_t)


    # ============================================================
    # DIRECTION DIAGNOSTIC
    # ============================================================
    print(f"\n\n{'='*60}")
    print(f"  DIRECTION TEST — {label_type}")
    print(f"{'='*60}")

    for stock in ['AAPL', 'MSFT', 'SPY']:
        if stock not in cnn_ticker_data:
            continue
        df_full, _, _ = cnn_ticker_data[stock]
        n = len(df_full)
        test_start = int(n * TRAIN_RATIO)
        df_test = df_full.iloc[test_start:].reset_index(drop=True)

        print(f"\n  --- {stock} ({len(df_test)} test bars) ---")
        print(f"  {'Strategy':<20} {'Type':<10} {'Thr':>5} {'N':>6} {'DA':>8} {'p':>10} {'Sig':>5}")

        for strat_name, (model, _, _) in all_models.items():
            cnn_feat = run_inference(df_test, model, sc_mean, sc_scale)
            results, base_da, base_n = direction_test(stock, df_test, cnn_feat, strat_name)

            for (sig_type, thr, n_trades, da, p_val, avg_ret) in results:
                if thr in [0.4, 0.5, 0.6]:  # 只显示关键阈值
                    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
                    print(f"  {strat_name:<20} {sig_type:<10} {thr:>5.1f} {n_trades:>6} "
                          f"{da:>8.4f} {p_val:>10.4f} {sig:>5}")

        print(f"  {'BASELINE':<20} {'DT→long':<10} {'all':>5} {base_n:>6} {base_da:>8.4f}")

    # Save best model
    print(f"\n\n  === Val Accuracy Summary ({label_type}) ===")
    print(f"  {'Strategy':<20} {'Bottom':>8} {'Top':>8} {'Avg':>8}")
    best_name, best_avg = None, 0
    for name, (_, ab, at) in all_models.items():
        avg = (ab + at) / 2
        print(f"  {name:<20} {ab:>8.1f} {at:>8.1f} {avg:>8.1f}")
        if avg > best_avg:
            best_avg, best_name = avg, name

    print(f"\n  Best: {best_name} ({best_avg:.1f}%)")

    # Save best
    best_model = all_models[best_name][0]
    save_dict = {
        'model_state_dict': best_model.state_dict(),
        'scaler_mean': sc_mean,
        'scaler_scale': sc_scale,
        'n_features': CNN_N_FEATURES,
        'window': CNN_WINDOW,
        'label_type': label_type,
        'strategy': best_name,
        'model_class': best_model.__class__.__name__,
    }
    save_path = os.path.join(SAVE_DIR, f"cnn_transfer_{label_type}_best.pt")
    torch.save(save_dict, save_path)
    print(f"  ✓ Saved to {save_path}")

In [ ]:
"""
双流CNN: 价格流 + 成交量流 独立卷积后融合
================================================================
问题: 单流CNN的conv1把24个特征混在一起, 模型无法分别推理
解决: 两个独立的卷积分支分别处理价格形态和成交量形态, 然后融合

架构:
  价格流 (16 feat) → conv1_p → conv2_p → pool → 64d
  成交量流 (8 feat) → conv1_v → conv2_v → pool → 32d
  融合: concat(64d, 32d) = 96d → fc_head → 方向预测

对比实验:
  A. 单流CNN 16feat (baseline)
  B. 单流CNN 24feat (上一轮结果)
  C. 双流CNN 16+8 (本轮)
  D. 双流CNN + 预训练价格流 (用strong模型初始化)
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIG
# ============================================================
DRIVE_BASE = CONFIG['data_dir'] / r'splits'
SAVE_DIR = CONFIG['data_dir'] / r'models'
FREQ_RAW = "1min"
RESAMPLE_PERIOD = 5
TICKERS = ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
TRAIN_RATIO = 0.8

CNN_WINDOW = 60
N_PRICE_FEAT = 16
N_VOL_FEAT = 8
CNN_BATCH = 64
TREND_PCT = 0.5
TREND_LOOKBACK = 18
CNN_LOOKAHEAD = 18

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")
print(f"Architecture: Dual-stream CNN ({N_PRICE_FEAT} price + {N_VOL_FEAT} volume)")


# ============================================================
# DATA LOADING (复用)
# ============================================================
def load_split_csv(csv_path):
    df = pd.read_csv(csv_path)
    col_map = {}
    for col in df.columns:
        cl = col.lower().strip()
        if cl == 'ts_event': col_map[col] = 'timestamp'
        elif cl == 'open': col_map[col] = 'open'
        elif cl == 'high': col_map[col] = 'high'
        elif cl == 'low': col_map[col] = 'low'
        elif cl == 'close': col_map[col] = 'close'
        elif cl == 'volume': col_map[col] = 'volume'
    df = df.rename(columns=col_map)
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df

def resample(df, period):
    df = df.set_index('timestamp').sort_index()
    resampled = df.resample(f'{period}min').agg({
        'open': 'first', 'high': 'max', 'low': 'min',
        'close': 'last', 'volume': 'sum'
    }).dropna(subset=['close'])
    return resampled.reset_index()

def load_ticker(ticker):
    data_dir = os.path.join(DRIVE_BASE, f"{ticker}_{FREQ_RAW}")
    dfs = []
    for split in ['train', 'test']:
        fpath = os.path.join(data_dir, f"{split}.csv")
        if os.path.exists(fpath):
            dfs.append(load_split_csv(fpath))
    if not dfs:
        return None
    df = pd.concat(dfs, ignore_index=True)
    df = df.sort_values('timestamp').reset_index(drop=True)
    df = resample(df, RESAMPLE_PERIOD)
    return df


# ============================================================
# FEATURES: 分别返回价格特征和成交量特征
# ============================================================
def compute_price_features(df):
    """原始16个价格/技术特征"""
    close = df['close'].values.astype(float)
    high = df['high'].values.astype(float)
    low = df['low'].values.astype(float)
    volume = df['volume'].values.astype(float)
    feat = pd.DataFrame()
    ret = pd.Series(close).pct_change()
    feat['ret_1'] = ret.values
    feat['ret_5'] = ret.rolling(5).sum().values
    feat['ret_15'] = ret.rolling(15).sum().values
    feat['high_low_range'] = (high - low) / (close + 1e-10)
    feat['close_position'] = (close - low) / (high - low + 1e-10)
    sma5 = pd.Series(close).rolling(5).mean()
    sma15 = pd.Series(close).rolling(15).mean()
    sma30 = pd.Series(close).rolling(30).mean()
    feat['ma5_15'] = ((sma5 - sma15) / (sma15 + 1e-10)).values
    feat['ma5_30'] = ((sma5 - sma30) / (sma30 + 1e-10)).values
    feat['vol_5'] = ret.rolling(5).std().values
    feat['vol_15'] = ret.rolling(15).std().values
    delta = ret.copy()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss_v = (-delta).clip(lower=0).rolling(14).mean()
    feat['rsi'] = (100 - 100 / (1 + gain / (loss_v + 1e-10))).values
    feat['vol_ratio'] = volume / (pd.Series(volume).rolling(20).mean().values + 1e-10)
    sma20 = pd.Series(close).rolling(20).mean()
    std20 = pd.Series(close).rolling(20).std()
    feat['bb_pos'] = ((close - sma20) / (2 * std20 + 1e-10)).values
    ema12 = pd.Series(close).ewm(span=12).mean()
    ema26 = pd.Series(close).ewm(span=26).mean()
    feat['macd'] = ((ema12 - ema26) / (close + 1e-10)).values
    tr = np.maximum(high - low, np.maximum(
        np.abs(high - np.roll(close, 1)), np.abs(low - np.roll(close, 1))))
    feat['atr'] = (pd.Series(tr).rolling(14).mean() / (close + 1e-10)).values
    feat['ret_3'] = ret.rolling(3).sum().values
    feat['close_pos_5'] = pd.Series(feat['close_position'].values).rolling(5).mean().values
    cols = list(feat.columns)[:N_PRICE_FEAT]
    return feat[cols].values.astype(np.float32)


def compute_volume_features(df):
    """8个成交量特征"""
    close = df['close'].values.astype(float)
    high = df['high'].values.astype(float)
    low = df['low'].values.astype(float)
    volume = df['volume'].values.astype(float)
    n = len(close)
    feat = pd.DataFrame(index=range(n))
    ret = pd.Series(close).pct_change()
    vol_s = pd.Series(volume)
    ret_s = pd.Series(ret.values)

    # 1. 相对成交量
    vol_sma20 = vol_s.rolling(20).mean()
    feat['rel_vol'] = volume / (vol_sma20.values + 1e-10)

    # 2. 成交量z-score
    vol_std20 = vol_s.rolling(20).std()
    feat['vol_zscore'] = (volume - vol_sma20.values) / (vol_std20.values + 1e-10)

    # 3. 反弹/下跌成交量比
    vol_recent = vol_s.rolling(6).mean()
    vol_prior = vol_s.shift(6).rolling(6).mean()
    feat['bounce_vol_ratio'] = vol_recent.values / (vol_prior.values + 1e-10)

    # 4. OFI代理
    ofi_raw = volume * ((close - low) - (high - close)) / (high - low + 1e-10)
    ofi_cum5 = pd.Series(ofi_raw).rolling(5).sum()
    ofi_std = pd.Series(ofi_raw).rolling(20).std()
    feat['ofi_proxy'] = ofi_cum5.values / (ofi_std.values + 1e-10)

    # 5. LMSW C2系数
    vol_log = np.log(volume / (vol_sma20.values + 1e-10) + 1e-10)
    vr_interaction = vol_log * ret.values
    vr_s = pd.Series(vr_interaction)
    ret_next = ret_s.shift(-1)
    feat['c2_coeff'] = vr_s.rolling(20).corr(ret_next).values

    # 6. OBV背离
    obv = np.cumsum(np.where(ret.values > 0, volume,
                    np.where(ret.values < 0, -volume, 0)))
    obv_slope = pd.Series(obv).diff(5) / 5
    price_slope = pd.Series(close).diff(5) / 5
    obv_slope_norm = obv_slope / (obv_slope.rolling(20).std() + 1e-10)
    price_slope_norm = price_slope / (price_slope.rolling(20).std() + 1e-10)
    feat['obv_div'] = (obv_slope_norm - price_slope_norm).values

    # 7. 成交量趋势斜率 (向量化版本, 避免慢循环)
    vol_norm = volume / (vol_sma20.values + 1e-10)
    vol_norm_s = pd.Series(vol_norm)
    # 用rolling corr with index来近似斜率
    idx_series = pd.Series(np.arange(n, dtype=float))
    # 简化: 用 (V_now - V_10ago) / 10 代替线性回归
    feat['vol_slope'] = (vol_norm_s - vol_norm_s.shift(10)).values / 10

    # 8. VPIN代理
    ret_std = ret_s.rolling(20).std()
    z = ret.values / (ret_std.values + 1e-10)
    from scipy.stats import norm
    buy_pct = norm.cdf(z)
    imbalance = np.abs(volume * buy_pct - volume * (1 - buy_pct))
    vpin = pd.Series(imbalance).rolling(20).sum() / (vol_s.rolling(20).sum() + 1e-10)
    feat['vpin_proxy'] = vpin.values

    cols = list(feat.columns)[:N_VOL_FEAT]
    return feat[cols].values.astype(np.float32)


# ============================================================
# MODELS
# ============================================================

class DualStreamCNN(nn.Module):
    """
    双流CNN: 价格流和成交量流独立卷积后融合

    价格流: (B, 1, 60, 16) → conv1(32) → conv2(64) → pool → 64d
    成交量流: (B, 1, 60, 8) → conv1(16) → conv2(32) → pool → 32d
    融合: concat → 96d → fc(48) → dropout → fc(1)
    """
    def __init__(self):
        super().__init__()
        # 价格流 (和原始CNN同架构)
        self.price_conv1 = nn.Conv2d(1, 32, kernel_size=(3, N_PRICE_FEAT), padding=(1, 0))
        self.price_conv2 = nn.Conv2d(32, 64, kernel_size=(3, 1), padding=(1, 0))
        self.price_pool = nn.AdaptiveAvgPool2d((1, 1))

        # 成交量流 (较小, 因为只有8个特征)
        self.vol_conv1 = nn.Conv2d(1, 16, kernel_size=(3, N_VOL_FEAT), padding=(1, 0))
        self.vol_conv2 = nn.Conv2d(16, 32, kernel_size=(3, 1), padding=(1, 0))
        self.vol_pool = nn.AdaptiveAvgPool2d((1, 1))

        # 融合层
        fused_dim = 64 + 32  # = 96
        self.fc_bottom = nn.Sequential(
            nn.Linear(fused_dim, 48),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(48, 1)
        )
        self.fc_top = nn.Sequential(
            nn.Linear(fused_dim, 48),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(48, 1)
        )

    def forward(self, x_price, x_vol):
        # 价格流
        p = torch.relu(self.price_conv1(x_price))
        p = torch.relu(self.price_conv2(p))
        p = self.price_pool(p).squeeze(-1).squeeze(-1)  # (B, 64)

        # 成交量流
        v = torch.relu(self.vol_conv1(x_vol))
        v = torch.relu(self.vol_conv2(v))
        v = self.vol_pool(v).squeeze(-1).squeeze(-1)  # (B, 32)

        # 融合
        fused = torch.cat([p, v], dim=1)  # (B, 96)
        return self.fc_bottom(fused).squeeze(-1), self.fc_top(fused).squeeze(-1)


class DualStreamCNN_Attention(nn.Module):
    """
    双流CNN + 交叉注意力: 让成交量流"审查"价格流的判断

    价格流: → 64d embedding
    成交量流: → 32d embedding
    交叉注意力: vol_query × price_key → attention weight → 加权价格特征
    """
    def __init__(self):
        super().__init__()
        # 价格流
        self.price_conv1 = nn.Conv2d(1, 32, kernel_size=(3, N_PRICE_FEAT), padding=(1, 0))
        self.price_conv2 = nn.Conv2d(32, 64, kernel_size=(3, 1), padding=(1, 0))
        self.price_pool = nn.AdaptiveAvgPool2d((1, 1))

        # 成交量流
        self.vol_conv1 = nn.Conv2d(1, 16, kernel_size=(3, N_VOL_FEAT), padding=(1, 0))
        self.vol_conv2 = nn.Conv2d(16, 32, kernel_size=(3, 1), padding=(1, 0))
        self.vol_pool = nn.AdaptiveAvgPool2d((1, 1))

        # 交叉注意力: 成交量gate价格
        self.gate = nn.Sequential(
            nn.Linear(32, 64),
            nn.Sigmoid()  # 0-1的gate, 控制价格信号的通过
        )

        # 融合: gated_price(64) + vol(32) = 96
        fused_dim = 64 + 32
        self.fc_bottom = nn.Sequential(
            nn.Linear(fused_dim, 48),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(48, 1)
        )
        self.fc_top = nn.Sequential(
            nn.Linear(fused_dim, 48),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(48, 1)
        )

    def forward(self, x_price, x_vol):
        p = torch.relu(self.price_conv1(x_price))
        p = torch.relu(self.price_conv2(p))
        p = self.price_pool(p).squeeze(-1).squeeze(-1)

        v = torch.relu(self.vol_conv1(x_vol))
        v = torch.relu(self.vol_conv2(v))
        v = self.vol_pool(v).squeeze(-1).squeeze(-1)

        # 成交量gate: 让成交量信号控制价格信号的强度
        # gate ≈ 1: 成交量确认价格信号 (放量反弹 → 信任)
        # gate ≈ 0: 成交量否定价格信号 (缩量反弹 → 抑制)
        gate = self.gate(v)  # (B, 64)
        gated_price = p * gate  # 元素级乘法

        fused = torch.cat([gated_price, v], dim=1)
        return self.fc_bottom(fused).squeeze(-1), self.fc_top(fused).squeeze(-1)


# ============================================================
# LABELS + DATASETS (双流版本)
# ============================================================
def find_trend_and_label(df, label_type='sustained'):
    close = df['close'].values.astype(float)
    n = len(close)
    trend_pct = TREND_PCT / 100.0
    LB = TREND_LOOKBACK; LA = CNN_LOOKAHEAD
    positions = []
    for t in range(max(CNN_WINDOW, LB), n - LA):
        p_now = close[t]; p_past = close[t - LB]
        if p_now <= 0 or p_past <= 0: continue
        move = (p_now - p_past) / p_past
        if move < -trend_pct:
            half = LA // 2
            label = 1 if np.mean(close[t + half:t + LA]) > p_now else 0
            positions.append({'bar_idx': t, 'trend_dir': 'down', 'label': label})
        elif move > trend_pct:
            half = LA // 2
            label = 1 if np.mean(close[t + half:t + LA]) < p_now else 0
            positions.append({'bar_idx': t, 'trend_dir': 'up', 'label': label})
    return positions


def build_dual_dataset(price_feat, vol_feat, positions):
    Xp, Xv, y = [], [], []
    for p in positions:
        idx = p['bar_idx']
        if idx < CNN_WINDOW: continue
        wp = price_feat[idx - CNN_WINDOW:idx]
        wv = vol_feat[idx - CNN_WINDOW:idx]
        if wp.shape != (CNN_WINDOW, N_PRICE_FEAT) or wv.shape != (CNN_WINDOW, N_VOL_FEAT):
            continue
        Xp.append(wp); Xv.append(wv); y.append(p['label'])
    return (np.array(Xp, dtype=np.float32),
            np.array(Xv, dtype=np.float32),
            np.array(y, dtype=np.int64))


def build_all_dual_datasets(all_data, label_type):
    down_p_tr, down_v_tr, down_y_tr = [], [], []
    down_p_v, down_v_v, down_y_v = [], [], []
    up_p_tr, up_v_tr, up_y_tr = [], [], []
    up_p_v, up_v_v, up_y_v = [], [], []

    for ticker, (df, pf, vf, tei) in all_data.items():
        positions = find_trend_and_label(df, label_type)
        pos_down = [p for p in positions if p['bar_idx'] < tei and p['trend_dir'] == 'down']
        pos_up = [p for p in positions if p['bar_idx'] < tei and p['trend_dir'] == 'up']

        for pos_list, (pTr, vTr, yTr), (pV, vV, yV) in [
            (pos_down, (down_p_tr, down_v_tr, down_y_tr), (down_p_v, down_v_v, down_y_v)),
            (pos_up, (up_p_tr, up_v_tr, up_y_tr), (up_p_v, up_v_v, up_y_v))
        ]:
            if len(pos_list) > 30:
                Xp, Xv, y = build_dual_dataset(pf, vf, pos_list)
                if len(Xp) > 10:
                    vs = max(int(len(Xp) * 0.15), 1)
                    pTr.append(Xp[:-vs]); vTr.append(Xv[:-vs]); yTr.append(y[:-vs])
                    pV.append(Xp[-vs:]); vV.append(Xv[-vs:]); yV.append(y[-vs:])

    # Concat
    dp_tr = np.concatenate(down_p_tr); dv_tr = np.concatenate(down_v_tr); dy_tr = np.concatenate(down_y_tr)
    dp_v = np.concatenate(down_p_v);   dv_v = np.concatenate(down_v_v);   dy_v = np.concatenate(down_y_v)
    up_tr = np.concatenate(up_p_tr);   uv_tr = np.concatenate(up_v_tr);   uy_tr = np.concatenate(up_y_tr)
    up_v = np.concatenate(up_p_v);     uv_v = np.concatenate(up_v_v);     uy_v = np.concatenate(up_y_v)

    # Scalers (分别标准化)
    all_p = np.concatenate([dp_tr, up_tr])
    all_v = np.concatenate([dv_tr, uv_tr])

    sc_p = StandardScaler().fit(all_p.reshape(-1, N_PRICE_FEAT))
    sc_v = StandardScaler().fit(all_v.reshape(-1, N_VOL_FEAT))

    for arr in [dp_tr, dp_v, up_tr, up_v]:
        arr[:] = sc_p.transform(arr.reshape(-1, N_PRICE_FEAT)).reshape(arr.shape)
    for arr in [dv_tr, dv_v, uv_tr, uv_v]:
        arr[:] = sc_v.transform(arr.reshape(-1, N_VOL_FEAT)).reshape(arr.shape)

    # NaN cleanup
    for arr in [dp_tr, dp_v, up_tr, up_v, dv_tr, dv_v, uv_tr, uv_v]:
        arr[np.isnan(arr)] = 0; arr[np.isinf(arr)] = 0

    return (dp_tr, dv_tr, dy_tr, dp_v, dv_v, dy_v,
            up_tr, uv_tr, uy_tr, up_v, uv_v, uy_v,
            sc_p.mean_.astype(np.float32), sc_p.scale_.astype(np.float32),
            sc_v.mean_.astype(np.float32), sc_v.scale_.astype(np.float32))


# ============================================================
# TRAINING (双流版本)
# ============================================================
def train_dual(model, data, name, lr=1e-3, epochs=60, patience=12):
    (dp_tr, dv_tr, dy_tr, dp_v, dv_v, dy_v,
     up_tr, uv_tr, uy_tr, up_v, uv_v, uy_v, *_) = data

    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                           lr=lr, weight_decay=1e-5)
    crit = nn.BCEWithLogitsLoss()

    # DataLoaders
    ds_d = TensorDataset(
        torch.FloatTensor(dp_tr).unsqueeze(1),
        torch.FloatTensor(dv_tr).unsqueeze(1),
        torch.FloatTensor(dy_tr.astype(np.float32)))
    dl_d = DataLoader(ds_d, batch_size=CNN_BATCH, shuffle=True)

    ds_u = TensorDataset(
        torch.FloatTensor(up_tr).unsqueeze(1),
        torch.FloatTensor(uv_tr).unsqueeze(1),
        torch.FloatTensor(uy_tr.astype(np.float32)))
    dl_u = DataLoader(ds_u, batch_size=CNN_BATCH, shuffle=True)

    # Val tensors
    dpv = torch.FloatTensor(dp_v).unsqueeze(1).to(DEVICE)
    dvv = torch.FloatTensor(dv_v).unsqueeze(1).to(DEVICE)
    dyv = torch.FloatTensor(dy_v.astype(np.float32)).to(DEVICE)
    upv = torch.FloatTensor(up_v).unsqueeze(1).to(DEVICE)
    uvv = torch.FloatTensor(uv_v).unsqueeze(1).to(DEVICE)
    uyv = torch.FloatTensor(uy_v.astype(np.float32)).to(DEVICE)

    best_vl, best_st, pat = float('inf'), None, 0
    for ep in range(epochs):
        model.train()
        for xp, xv, yb in dl_d:
            xp, xv, yb = xp.to(DEVICE), xv.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            out = model(xp, xv)[0]
            loss = crit(out, yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        for xp, xv, yb in dl_u:
            xp, xv, yb = xp.to(DEVICE), xv.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            out = model(xp, xv)[1]
            loss = crit(out, yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        with torch.no_grad():
            vl_b = crit(model(dpv, dvv)[0], dyv).item()
            vl_t = crit(model(upv, uvv)[1], uyv).item()
            vl = (vl_b + vl_t) / 2

        if vl < best_vl:
            best_vl, best_st, pat = vl, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            pat += 1
            if pat >= patience: break

        if (ep + 1) % 10 == 0:
            print(f"    [{name}] Epoch {ep+1}: val_loss={vl:.4f}")

    if best_st: model.load_state_dict(best_st)
    model = model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        pb = torch.sigmoid(model(dpv, dvv)[0]).cpu().numpy()
        pt = torch.sigmoid(model(upv, uvv)[1]).cpu().numpy()
    acc_b = np.mean((pb > 0.5).astype(int) == dy_v) * 100
    acc_t = np.mean((pt > 0.5).astype(int) == uy_v) * 100

    stop_ep = ep + 1 - patience if pat >= patience else ep + 1
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  [{name}] Val Acc: Bottom={acc_b:.1f}%, Top={acc_t:.1f}% | params={n_params} | early_stop={stop_ep}")
    return model, acc_b, acc_t


# ============================================================
# INFERENCE (双流)
# ============================================================
@torch.no_grad()
def run_dual_inference(df, model, sc_p_mean, sc_p_scale, sc_v_mean, sc_v_scale):
    pf = compute_price_features(df)
    vf = compute_volume_features(df)

    pf_norm = (pf - sc_p_mean) / (sc_p_scale + 1e-8)
    vf_norm = (vf - sc_v_mean) / (sc_v_scale + 1e-8)
    pf_norm = np.nan_to_num(pf_norm, nan=0.0, posinf=0.0, neginf=0.0)
    vf_norm = np.nan_to_num(vf_norm, nan=0.0, posinf=0.0, neginf=0.0)

    prob_bottom = np.full(len(df), 0.5)
    prob_top = np.full(len(df), 0.5)

    batch_size = 512
    for start in range(CNN_WINDOW, len(pf_norm), batch_size):
        end = min(start + batch_size, len(pf_norm))
        bp, bv, bi = [], [], []
        for i in range(start, end):
            wp = pf_norm[i - CNN_WINDOW:i]
            wv = vf_norm[i - CNN_WINDOW:i]
            if wp.shape == (CNN_WINDOW, N_PRICE_FEAT) and wv.shape == (CNN_WINDOW, N_VOL_FEAT):
                bp.append(wp); bv.append(wv); bi.append(i)
        if not bp: continue
        xp = torch.FloatTensor(np.array(bp)).unsqueeze(1).to(DEVICE)
        xv = torch.FloatTensor(np.array(bv)).unsqueeze(1).to(DEVICE)
        pb, pt = model(xp, xv)
        for j, idx in enumerate(bi):
            prob_bottom[idx] = torch.sigmoid(pb[j]).item()
            prob_top[idx] = torch.sigmoid(pt[j]).item()

    return pd.DataFrame({
        'cnn_prob_bottom': prob_bottom,
        'cnn_prob_top': prob_top
    }, index=df.index)


def direction_test(stock, df, cnn_features):
    close = df['close'].values
    n = len(close)
    trend_pct = TREND_PCT / 100.0
    LA = CNN_LOOKAHEAD
    future_ret = pd.Series(close).pct_change(LA).shift(-LA)
    downtrend = np.zeros(n, dtype=bool)
    uptrend = np.zeros(n, dtype=bool)
    for t in range(TREND_LOOKBACK, n):
        move = (close[t] - close[t - TREND_LOOKBACK]) / (close[t - TREND_LOOKBACK] + 1e-10)
        if move < -trend_pct: downtrend[t] = True
        elif move > trend_pct: uptrend[t] = True

    results = []
    for thr in [0.3, 0.4, 0.5, 0.6, 0.7]:
        mask = downtrend & (cnn_features['cnn_prob_bottom'] > thr)
        rets = future_ret[mask].dropna()
        if len(rets) >= 10:
            n_up = int((rets > 0).sum())
            da = n_up / len(rets)
            p = stats.binomtest(n_up, len(rets), 0.5, alternative='greater').pvalue
            sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
            results.append(('DT+Bot', thr, len(rets), da, p, sig, rets.mean()))

        mask = uptrend & (cnn_features['cnn_prob_top'] > thr)
        rets = future_ret[mask].dropna()
        if len(rets) >= 10:
            n_down = int((rets < 0).sum())
            da = n_down / len(rets)
            p = stats.binomtest(n_down, len(rets), 0.5, alternative='greater').pvalue
            sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
            results.append(('UT+Top', thr, len(rets), da, p, sig, rets.mean()))

    base_rets = future_ret[downtrend].dropna()
    base_da = (base_rets > 0).mean() if len(base_rets) > 0 else 0.5
    return results, base_da, len(base_rets)


# ============================================================
# MAIN
# ============================================================
print(f"\n{'#'*60}")
print(f"# LOADING DATA")
print(f"{'#'*60}")

all_data = {}
for ticker in TICKERS:
    df = load_ticker(ticker)
    if df is None: continue
    n = len(df)
    tei = int(n * TRAIN_RATIO)
    pf = compute_price_features(df)
    vf = compute_volume_features(df)
    all_data[ticker] = (df, pf, vf, tei)
    print(f"  {ticker}: {n} bars")


# Build datasets
print(f"\n{'#'*60}")
print(f"# BUILDING DATASETS (sustained label)")
print(f"{'#'*60}")

data = build_all_dual_datasets(all_data, 'sustained')
(dp_tr, dv_tr, dy_tr, dp_v, dv_v, dy_v,
 up_tr, uv_tr, uy_tr, up_v, uv_v, uy_v,
 sc_p_mean, sc_p_scale, sc_v_mean, sc_v_scale) = data

print(f"  Bottom: train={len(dp_tr)}, val={len(dp_v)}, pos_rate={dy_tr.mean():.3f}")
print(f"  Top:    train={len(up_tr)}, val={len(up_v)}, pos_rate={uy_tr.mean():.3f}")

results = {}

# --- Model C: DualStream CNN ---
print(f"\n{'='*60}")
print(f"  MODEL C: Dual-Stream CNN (concat fusion)")
print(f"{'='*60}")
torch.manual_seed(SEED)
model_C = DualStreamCNN().to(DEVICE)
model_C, accC_b, accC_t = train_dual(model_C, data, "dual_concat", lr=1e-3)
results['dual_concat'] = (model_C, accC_b, accC_t)

# --- Model D: DualStream CNN + Attention Gate ---
print(f"\n{'='*60}")
print(f"  MODEL D: Dual-Stream CNN (attention gate)")
print(f"{'='*60}")
torch.manual_seed(SEED)
model_D = DualStreamCNN_Attention().to(DEVICE)
model_D, accD_b, accD_t = train_dual(model_D, data, "dual_attn", lr=1e-3)
results['dual_attn'] = (model_D, accD_b, accD_t)

# --- Model E: DualStream + pretrained price stream ---
print(f"\n{'='*60}")
print(f"  MODEL E: Dual-Stream + pretrained price conv (from strong)")
print(f"{'='*60}")
strong_path = os.path.join(SAVE_DIR, "cnn_dual_5min_strong.pt")
if os.path.exists(strong_path):
    strong_ckpt = torch.load(strong_path, map_location='cpu', weights_only=True)
    strong_state = strong_ckpt['model_state_dict']

    torch.manual_seed(SEED)
    model_E = DualStreamCNN_Attention().to(DEVICE)
    # 加载预训练的价格conv层
    model_E.price_conv1.load_state_dict({
        k.replace('conv1.', ''): v for k, v in strong_state.items() if k.startswith('conv1.')
    })
    model_E.price_conv2.load_state_dict({
        k.replace('conv2.', ''): v for k, v in strong_state.items() if k.startswith('conv2.')
    })
    # 冻结价格conv
    model_E.price_conv1.requires_grad_(False)
    model_E.price_conv2.requires_grad_(False)

    model_E, accE_b, accE_t = train_dual(model_E, data, "dual_pretrain", lr=1e-3)
    results['dual_pretrain'] = (model_E, accE_b, accE_t)
else:
    print(f"  ⚠ Strong model not found, skipping")


# ============================================================
# DIRECTION TEST
# ============================================================
print(f"\n\n{'#'*60}")
print(f"# DIRECTION TEST")
print(f"{'#'*60}")

for stock in ['AAPL', 'MSFT', 'SPY']:
    if stock not in all_data: continue
    df_full, _, _, _ = all_data[stock]
    n = len(df_full)
    test_start = int(n * TRAIN_RATIO)
    df_test = df_full.iloc[test_start:].reset_index(drop=True)

    print(f"\n{'='*60}")
    print(f"  {stock} ({len(df_test)} test bars)")
    print(f"{'='*60}")
    print(f"  {'Model':<16} {'Type':<10} {'Thr':>5} {'N':>6} {'DA':>8} {'p':>10} {'Sig':>5}")

    for name, (model, _, _) in results.items():
        feat = run_dual_inference(df_test, model, sc_p_mean, sc_p_scale, sc_v_mean, sc_v_scale)
        res, base_da, base_n = direction_test(stock, df_test, feat)

        for (sig_type, thr, n_trades, da, p_val, sig, avg_ret) in res:
            print(f"  {name:<16} {sig_type:<10} {thr:>5.1f} {n_trades:>6} "
                  f"{da:>8.4f} {p_val:>10.4f} {sig:>5}")

    print(f"  {'BASELINE':<16} {'DT→long':<10} {'all':>5} {base_n:>6} {base_da:>8.4f}")


# Summary
print(f"\n\n{'='*60}")
print(f"  SUMMARY")
print(f"{'='*60}")
print(f"  Previous results:     16feat=51.4%, 24feat_single=51.5%")
print(f"  {'Model':<16} {'Bottom':>8} {'Top':>8} {'Avg':>8}")
for name, (_, ab, at) in results.items():
    print(f"  {name:<16} {ab:>8.1f} {at:>8.1f} {(ab+at)/2:>8.1f}")

# Save best
best_name = max(results, key=lambda k: (results[k][1] + results[k][2]) / 2)
best_model = results[best_name][0]
save_dict = {
    'model_state_dict': best_model.state_dict(),
    'model_class': best_model.__class__.__name__,
    'sc_p_mean': sc_p_mean, 'sc_p_scale': sc_p_scale,
    'sc_v_mean': sc_v_mean, 'sc_v_scale': sc_v_scale,
    'n_price_feat': N_PRICE_FEAT, 'n_vol_feat': N_VOL_FEAT,
    'window': CNN_WINDOW, 'strategy': best_name,
}
save_path = os.path.join(SAVE_DIR, "cnn_dualstream_best.pt")
torch.save(save_dict, save_path)
print(f"\n  Best: {best_name}")
print(f"  ✓ Saved to {save_path}")

In [ ]:
"""
ATR障碍参数扫描: 找到pos_rate≈30%的配置
================================================================
问题: tp=1.5×ATR, sl=1.0×ATR + trail → pos=50% → CNN弱
目标: 找到ATR参数使pos_rate≈30%, 同时保持ATR自适应性

扫描:
  tp_mult: [2.0, 2.5, 3.0]     ← 更高止盈门槛
  sl_mult: [0.8, 1.0, 1.2]     ← 止损
  trailing: [on, off]           ← 跟踪止损是否启用
  trail_activate: [0.5, 0.7]
  trail_dist: [0.3, 0.5]
"""

import os, sys
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

DRIVE_BASE = CONFIG['data_dir'] / r'splits'
FREQ_RAW = "1min"; RESAMPLE_PERIOD = 5
TICKERS = ["AAPL", "NVDA", "TSLA", "SPY"]  # 代表性子集
TRAIN_RATIO = 0.8
TREND_LOOKBACK = 18; TREND_PCT = 0.005
ATR_PERIOD = 14; MAX_BARS = 18
SEED = 42
np.random.seed(SEED)

def load_split_csv(p):
    df = pd.read_csv(p)
    cm = {}
    for c in df.columns:
        cl = c.lower().strip()
        if cl == 'ts_event': cm[c] = 'timestamp'
        elif cl in ('open','high','low','close','volume'): cm[c] = cl
    df = df.rename(columns=cm)
    if 'timestamp' in df.columns: df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df

def resample(df, period):
    df = df.set_index('timestamp').sort_index()
    r = df.resample(f'{period}min').agg(
        {'open':'first','high':'max','low':'min','close':'last','volume':'sum'}
    ).dropna(subset=['close'])
    return r.reset_index()

def load_ticker(ticker):
    d = os.path.join(DRIVE_BASE, f"{ticker}_{FREQ_RAW}")
    dfs = []
    for s in ['train','test']:
        fp = os.path.join(d, f"{s}.csv")
        if os.path.exists(fp): dfs.append(load_split_csv(fp))
    if not dfs: return None
    return resample(pd.concat(dfs, ignore_index=True).sort_values('timestamp').reset_index(drop=True), RESAMPLE_PERIOD)

def compute_atr(high, low, close, period=14):
    n = len(close); tr = np.zeros(n)
    tr[0] = high[0] - low[0]
    for i in range(1, n):
        tr[i] = max(high[i]-low[i], abs(high[i]-close[i-1]), abs(low[i]-close[i-1]))
    atr = np.full(n, np.nan)
    atr[period] = np.mean(tr[1:period+1])
    for i in range(period+1, n):
        atr[i] = (atr[i-1]*(period-1) + tr[i]) / period
    return atr

def atr_barrier(close, high, low, atr, idx, direction,
                tp_mult, sl_mult, max_bars,
                use_trailing=False, trail_activate=0.7, trail_dist=0.5):
    n = len(close)
    if np.isnan(atr[idx]) or atr[idx] < 1e-10: return -1  # skip
    entry = close[idx]; av = atr[idx]

    if direction == 'bottom':
        tp_p = entry + tp_mult * av
        sl_p = entry - sl_mult * av
        trail_trig = entry + trail_activate * tp_mult * av
        trail_d = trail_dist * av
        trailing = False; best = entry
        for t in range(idx+1, min(idx+max_bars+1, n)):
            # 止损永远优先
            if low[t] <= sl_p: return 0
            if high[t] > best: best = high[t]
            if use_trailing and not trailing and high[t] >= trail_trig:
                trailing = True
            if trailing:
                trail_tp = best - trail_d
                if low[t] <= trail_tp and trail_tp > entry:
                    return 1  # 锁利
            else:
                if high[t] >= tp_p: return 1
        return 0  # timeout = lose
    else:
        tp_p = entry - tp_mult * av
        sl_p = entry + sl_mult * av
        trail_trig = entry - trail_activate * tp_mult * av
        trail_d = trail_dist * av
        trailing = False; best = entry
        for t in range(idx+1, min(idx+max_bars+1, n)):
            if high[t] >= sl_p: return 0
            if low[t] < best: best = low[t]
            if use_trailing and not trailing and low[t] <= trail_trig:
                trailing = True
            if trailing:
                trail_tp = best + trail_d
                if high[t] >= trail_tp and trail_tp < entry:
                    return 1
            else:
                if low[t] <= tp_p: return 1
        return 0

# Load data
print("Loading data...")
all_data = {}
for ticker in TICKERS:
    df = load_ticker(ticker)
    if df is None: continue
    c = df['close'].values.astype(float)
    h = df['high'].values.astype(float)
    l = df['low'].values.astype(float)
    atr = compute_atr(h, l, c, ATR_PERIOD)
    tei = int(len(df) * TRAIN_RATIO)
    all_data[ticker] = (c, h, l, atr, tei)
    print(f"  {ticker}: {len(df)} bars")

# 收集趋势bar
print("\nCollecting trend bars...")
trend_bars = {'bottom': [], 'top': []}
for ticker, (c, h, l, atr, tei) in all_data.items():
    for t in range(120, tei):  # only train set
        if np.isnan(atr[t]): continue
        move = (c[t] - c[t-TREND_LOOKBACK]) / (c[t-TREND_LOOKBACK] + 1e-10)
        if move < -TREND_PCT:
            trend_bars['bottom'].append((ticker, t))
        elif move > TREND_PCT:
            trend_bars['top'].append((ticker, t))

print(f"  bottom: {len(trend_bars['bottom'])}, top: {len(trend_bars['top'])}")

# 采样 (用子集加速)
SAMPLE = 20000
for d in ['bottom', 'top']:
    if len(trend_bars[d]) > SAMPLE:
        idx = np.random.choice(len(trend_bars[d]), SAMPLE, replace=False)
        trend_bars[d] = [trend_bars[d][i] for i in idx]
    print(f"  {d} sampled: {len(trend_bars[d])}")


# ============================================================
# PARAMETER SWEEP
# ============================================================
print(f"\n{'='*80}")
print(f"PARAMETER SWEEP")
print(f"{'='*80}")

configs = []

# 不带跟踪止损
for tp in [1.5, 2.0, 2.5, 3.0]:
    for sl in [0.8, 1.0, 1.2]:
        configs.append({
            'tp': tp, 'sl': sl, 'trail': False,
            'trail_act': 0, 'trail_dist': 0,
            'name': f"tp{tp}_sl{sl}_notrail"
        })

# 带跟踪止损
for tp in [2.0, 2.5, 3.0]:
    for sl in [0.8, 1.0]:
        for ta in [0.5, 0.7]:
            for td in [0.3, 0.5]:
                configs.append({
                    'tp': tp, 'sl': sl, 'trail': True,
                    'trail_act': ta, 'trail_dist': td,
                    'name': f"tp{tp}_sl{sl}_trail{ta}_{td}"
                })

# 也加入v2的等效配置作为参考
# v2: tp=0.5%, sl=0.3% 对于ATR≈0.3的股票, 大约是 tp=1.67×ATR, sl=1.0×ATR

print(f"\n  Testing {len(configs)} configurations...")
print(f"\n  {'Config':<35} {'Bot_pos':>8} {'Top_pos':>8} {'Avg_pos':>8} {'Target':>8}")
print(f"  {'-'*75}")

good_configs = []
for cfg in configs:
    labels = {'bottom': [], 'top': []}
    for direction in ['bottom', 'top']:
        for ticker, t in trend_bars[direction]:
            c, h, l, atr, tei = all_data[ticker]
            lab = atr_barrier(c, h, l, atr, t, direction,
                              cfg['tp'], cfg['sl'], MAX_BARS,
                              cfg['trail'], cfg['trail_act'], cfg['trail_dist'])
            if lab >= 0: labels[direction].append(lab)

    if len(labels['bottom']) < 100 or len(labels['top']) < 100: continue

    bp = np.mean(labels['bottom'])
    tp_r = np.mean(labels['top'])
    avg = (bp + tp_r) / 2

    marker = ""
    if 0.25 <= avg <= 0.40:
        marker = " ← GOOD"
        good_configs.append((cfg, bp, tp_r, avg))
    elif 0.20 <= avg <= 0.45:
        marker = " ← ok"

    # 只打印有意义的
    if avg < 0.15 or avg > 0.55: continue

    print(f"  {cfg['name']:<35} {bp:>7.3f} {tp_r:>7.3f} {avg:>7.3f}  {marker}")

# ============================================================
# TOP CONFIGS
# ============================================================
print(f"\n\n{'='*80}")
print(f"TOP CONFIGS (pos_rate 25-40%)")
print(f"{'='*80}")

good_configs.sort(key=lambda x: abs(x[3] - 0.32))  # 最接近32%
for cfg, bp, tp_r, avg in good_configs[:10]:
    rr = cfg['tp'] / cfg['sl']  # 盈亏比
    trail_str = f"trail@{cfg['trail_act']}/{cfg['trail_dist']}" if cfg['trail'] else "no trail"
    print(f"  {cfg['name']:<35} bot={bp:.3f} top={tp_r:.3f} avg={avg:.3f} RR={rr:.1f} {trail_str}")

print(f"""
\n推荐:
  - 选avg≈0.30-0.35的配置 (和v2相似)
  - 盈亏比(RR) > 1.5 保证正EV
  - 有跟踪的配置通常pos稍高但RR更大 (让利润跑)
  - 最终: CNN用此标签训练应恢复到70%+ accuracy
""")

In [ ]:
"""
CNN拐点预测可视化
==========================================
生成图表:
1. 价格走势 + CNN bottom/top概率信号叠加
2. 高置信度拐点标注
3. 不同阈值下的信号分布
4. 真反转 vs 假反弹对比

用于论文展示和结果理解
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIG
# ============================================================
DRIVE_BASE = CONFIG['data_dir'] / r'splits'
MODEL_DIR = CONFIG['data_dir'] / r'models'
SAVE_FIG_DIR = CONFIG['data_dir'] / r'figures'
os.makedirs(SAVE_FIG_DIR, exist_ok=True)

FREQ_RAW = "1min"
RESAMPLE_PERIOD = 5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CNN_WINDOW = 60
N_PRICE_FEAT = 16
N_VOL_FEAT = 8
TREND_PCT = 0.5
TREND_LOOKBACK = 18
CNN_LOOKAHEAD = 18
TRAIN_RATIO = 0.8

plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 10


# ============================================================
# DATA + FEATURES (复用)
# ============================================================
def load_split_csv(csv_path):
    df = pd.read_csv(csv_path)
    col_map = {}
    for col in df.columns:
        cl = col.lower().strip()
        if cl == 'ts_event': col_map[col] = 'timestamp'
        elif cl == 'open': col_map[col] = 'open'
        elif cl == 'high': col_map[col] = 'high'
        elif cl == 'low': col_map[col] = 'low'
        elif cl == 'close': col_map[col] = 'close'
        elif cl == 'volume': col_map[col] = 'volume'
    df = df.rename(columns=col_map)
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df

def resample(df, period):
    df = df.set_index('timestamp').sort_index()
    resampled = df.resample(f'{period}min').agg({
        'open': 'first', 'high': 'max', 'low': 'min',
        'close': 'last', 'volume': 'sum'
    }).dropna(subset=['close'])
    return resampled.reset_index()

def load_ticker(ticker):
    data_dir = os.path.join(DRIVE_BASE, f"{ticker}_{FREQ_RAW}")
    dfs = []
    for split in ['train', 'test']:
        fpath = os.path.join(data_dir, f"{split}.csv")
        if os.path.exists(fpath):
            dfs.append(load_split_csv(fpath))
    if not dfs: return None
    df = pd.concat(dfs, ignore_index=True)
    df = df.sort_values('timestamp').reset_index(drop=True)
    df = resample(df, RESAMPLE_PERIOD)
    return df

def compute_price_features(df):
    close = df['close'].values.astype(float)
    high = df['high'].values.astype(float)
    low = df['low'].values.astype(float)
    volume = df['volume'].values.astype(float)
    feat = pd.DataFrame()
    ret = pd.Series(close).pct_change()
    feat['ret_1'] = ret.values
    feat['ret_5'] = ret.rolling(5).sum().values
    feat['ret_15'] = ret.rolling(15).sum().values
    feat['high_low_range'] = (high - low) / (close + 1e-10)
    feat['close_position'] = (close - low) / (high - low + 1e-10)
    sma5 = pd.Series(close).rolling(5).mean()
    sma15 = pd.Series(close).rolling(15).mean()
    sma30 = pd.Series(close).rolling(30).mean()
    feat['ma5_15'] = ((sma5 - sma15) / (sma15 + 1e-10)).values
    feat['ma5_30'] = ((sma5 - sma30) / (sma30 + 1e-10)).values
    feat['vol_5'] = ret.rolling(5).std().values
    feat['vol_15'] = ret.rolling(15).std().values
    delta = ret.copy()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss_v = (-delta).clip(lower=0).rolling(14).mean()
    feat['rsi'] = (100 - 100 / (1 + gain / (loss_v + 1e-10))).values
    feat['vol_ratio'] = volume / (pd.Series(volume).rolling(20).mean().values + 1e-10)
    sma20 = pd.Series(close).rolling(20).mean()
    std20 = pd.Series(close).rolling(20).std()
    feat['bb_pos'] = ((close - sma20) / (2 * std20 + 1e-10)).values
    ema12 = pd.Series(close).ewm(span=12).mean()
    ema26 = pd.Series(close).ewm(span=26).mean()
    feat['macd'] = ((ema12 - ema26) / (close + 1e-10)).values
    tr = np.maximum(high - low, np.maximum(
        np.abs(high - np.roll(close, 1)), np.abs(low - np.roll(close, 1))))
    feat['atr'] = (pd.Series(tr).rolling(14).mean() / (close + 1e-10)).values
    feat['ret_3'] = ret.rolling(3).sum().values
    feat['close_pos_5'] = pd.Series(feat['close_position'].values).rolling(5).mean().values
    cols = list(feat.columns)[:N_PRICE_FEAT]
    return feat[cols].values.astype(np.float32)


# ============================================================
# MODEL (用strong模型, 81%反弹检测)
# ============================================================
class CNNDualModel(nn.Module):
    def __init__(self, n_features=16, window=60):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=(3, n_features), padding=(1, 0))
        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3, 1), padding=(1, 0))
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc_bottom = nn.Linear(64, 1)
        self.fc_top = nn.Linear(64, 1)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1).squeeze(-1)
        return self.fc_bottom(x).squeeze(-1), self.fc_top(x).squeeze(-1)


@torch.no_grad()
def run_inference(df, model, sc_mean, sc_scale):
    features = compute_price_features(df)
    features_normed = (features - sc_mean) / (sc_scale + 1e-8)
    features_normed = np.nan_to_num(features_normed, nan=0.0, posinf=0.0, neginf=0.0)

    prob_bottom = np.full(len(df), np.nan)
    prob_top = np.full(len(df), np.nan)

    batch_size = 512
    for start in range(CNN_WINDOW, len(features_normed), batch_size):
        end = min(start + batch_size, len(features_normed))
        batch_w, batch_idx = [], []
        for i in range(start, end):
            w = features_normed[i - CNN_WINDOW:i]
            if w.shape == (CNN_WINDOW, N_PRICE_FEAT):
                batch_w.append(w); batch_idx.append(i)
        if not batch_w: continue
        x = torch.FloatTensor(np.array(batch_w)).unsqueeze(1).to(DEVICE)
        pb, pt = model(x)
        for j, idx in enumerate(batch_idx):
            prob_bottom[idx] = torch.sigmoid(pb[j]).item()
            prob_top[idx] = torch.sigmoid(pt[j]).item()

    df_out = df.copy()
    df_out['prob_bottom'] = prob_bottom
    df_out['prob_top'] = prob_top
    return df_out


# ============================================================
# VISUALIZATION FUNCTIONS
# ============================================================

def plot_price_with_signals(df, stock, start_idx, n_bars=500, thr=0.6,
                            save_path=None):
    """
    图1: 价格走势 + CNN信号叠加
    上面板: 价格 + 标注的拐点
    下面板: CNN bottom/top概率
    """
    end_idx = min(start_idx + n_bars, len(df))
    seg = df.iloc[start_idx:end_idx].copy()

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8),
                                    gridspec_kw={'height_ratios': [3, 1]},
                                    sharex=True)

    # --- 上面板: 价格 ---
    x = range(len(seg))
    ax1.plot(x, seg['close'].values, color='#333333', linewidth=0.8, label='Close')

    # 标注趋势
    close = seg['close'].values
    trend_pct = TREND_PCT / 100.0
    for t in range(TREND_LOOKBACK, len(seg)):
        move = (close[t] - close[t - TREND_LOOKBACK]) / (close[t - TREND_LOOKBACK] + 1e-10)
        if move < -trend_pct:
            ax1.axvspan(t - 0.5, t + 0.5, alpha=0.05, color='red')
        elif move > trend_pct:
            ax1.axvspan(t - 0.5, t + 0.5, alpha=0.05, color='green')

    # 标注高置信度bottom信号
    bot_mask = seg['prob_bottom'].values > thr
    down_mask = np.zeros(len(seg), dtype=bool)
    for t in range(TREND_LOOKBACK, len(seg)):
        move = (close[t] - close[t - TREND_LOOKBACK]) / (close[t - TREND_LOOKBACK] + 1e-10)
        if move < -trend_pct: down_mask[t] = True

    signal_bot = bot_mask & down_mask
    bot_idx = np.where(signal_bot)[0]
    if len(bot_idx) > 0:
        ax1.scatter(bot_idx, close[bot_idx], marker='^', color='#2196F3',
                   s=80, zorder=5, label=f'Bottom signal (p>{thr})')

    # 标注高置信度top信号
    top_mask = seg['prob_top'].values > thr
    up_mask = np.zeros(len(seg), dtype=bool)
    for t in range(TREND_LOOKBACK, len(seg)):
        move = (close[t] - close[t - TREND_LOOKBACK]) / (close[t - TREND_LOOKBACK] + 1e-10)
        if move > trend_pct: up_mask[t] = True

    signal_top = top_mask & up_mask
    top_idx = np.where(signal_top)[0]
    if len(top_idx) > 0:
        ax1.scatter(top_idx, close[top_idx], marker='v', color='#F44336',
                   s=80, zorder=5, label=f'Top signal (p>{thr})')

    ax1.set_ylabel('Price ($)')
    ax1.set_title(f'{stock} — CNN Turning Point Detection (5min bars, threshold={thr})',
                  fontsize=13, fontweight='bold')
    ax1.legend(loc='upper left', fontsize=9)
    ax1.grid(True, alpha=0.3)

    # --- 下面板: CNN概率 ---
    ax2.fill_between(x, 0.5, seg['prob_bottom'].values,
                     where=seg['prob_bottom'].values > 0.5,
                     alpha=0.4, color='#2196F3', label='P(bottom)')
    ax2.fill_between(x, seg['prob_top'].values, 0.5,
                     where=seg['prob_top'].values > 0.5,
                     alpha=0.4, color='#F44336', label='P(top)')
    ax2.axhline(y=0.5, color='gray', linewidth=0.5, linestyle='--')
    ax2.axhline(y=thr, color='blue', linewidth=0.5, linestyle=':', alpha=0.5)
    ax2.set_ylabel('CNN Probability')
    ax2.set_xlabel('Bar index')
    ax2.set_ylim(0, 1)
    ax2.legend(loc='upper right', fontsize=8)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
        print(f"  ✓ Saved: {save_path}")
    plt.show()


def plot_signal_outcomes(df, stock, save_path=None):
    """
    图2: 信号触发后的价格走势 (真反转 vs 假反弹)
    每个bottom信号后18bar的价格轨迹叠加
    """
    close = df['close'].values
    n = len(close)
    trend_pct = TREND_PCT / 100.0
    LA = CNN_LOOKAHEAD
    thr = 0.5

    # 找到所有trend-filtered bottom信号
    trajectories_up = []    # 真反转 (后续涨)
    trajectories_down = []  # 假反弹 (后续跌)

    for t in range(max(CNN_WINDOW, TREND_LOOKBACK), n - LA):
        move = (close[t] - close[t - TREND_LOOKBACK]) / (close[t - TREND_LOOKBACK] + 1e-10)
        if move < -trend_pct and df['prob_bottom'].iloc[t] > thr:
            # 归一化后续轨迹
            future = close[t:t + LA + 1]
            future_norm = (future - future[0]) / (future[0] + 1e-10) * 100  # %变化
            if close[t + LA] > close[t]:
                trajectories_up.append(future_norm)
            else:
                trajectories_down.append(future_norm)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # --- 左: 真反转轨迹 ---
    ax = axes[0]
    for traj in trajectories_up[:100]:  # 最多画100条
        ax.plot(traj, color='#2196F3', alpha=0.1, linewidth=0.5)
    if trajectories_up:
        avg_up = np.mean(trajectories_up, axis=0)
        ax.plot(avg_up, color='#1565C0', linewidth=2, label=f'Mean (n={len(trajectories_up)})')
    ax.axhline(y=0, color='gray', linewidth=0.5, linestyle='--')
    ax.set_title('True Reversals (price goes up)', fontweight='bold')
    ax.set_xlabel('Bars after signal')
    ax.set_ylabel('Price change (%)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # --- 中: 假反弹轨迹 ---
    ax = axes[1]
    for traj in trajectories_down[:100]:
        ax.plot(traj, color='#F44336', alpha=0.1, linewidth=0.5)
    if trajectories_down:
        avg_down = np.mean(trajectories_down, axis=0)
        ax.plot(avg_down, color='#C62828', linewidth=2, label=f'Mean (n={len(trajectories_down)})')
    ax.axhline(y=0, color='gray', linewidth=0.5, linestyle='--')
    ax.set_title('False Bounces (price goes down)', fontweight='bold')
    ax.set_xlabel('Bars after signal')
    ax.set_ylabel('Price change (%)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # --- 右: 两者对比 ---
    ax = axes[2]
    if trajectories_up:
        ax.plot(avg_up, color='#2196F3', linewidth=2, label=f'True reversal (n={len(trajectories_up)})')
    if trajectories_down:
        ax.plot(avg_down, color='#F44336', linewidth=2, label=f'False bounce (n={len(trajectories_down)})')
    ax.axhline(y=0, color='gray', linewidth=0.5, linestyle='--')
    ax.set_title('Comparison: Reversal vs Bounce', fontweight='bold')
    ax.set_xlabel('Bars after signal')
    ax.set_ylabel('Price change (%)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    total = len(trajectories_up) + len(trajectories_down)
    pct_up = len(trajectories_up) / total * 100 if total > 0 else 0

    plt.suptitle(f'{stock} — Bottom Signal Outcomes (thr={thr}, '
                 f'{len(trajectories_up)}/{total} = {pct_up:.1f}% true reversals)',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
        print(f"  ✓ Saved: {save_path}")
    plt.show()


def plot_probability_distribution(df, stock, save_path=None):
    """
    图3: CNN概率分布 (下跌趋势中 vs 非趋势)
    """
    close = df['close'].values
    trend_pct = TREND_PCT / 100.0

    down_probs = []
    up_probs = []
    neutral_probs = []

    for t in range(TREND_LOOKBACK, len(df)):
        if np.isnan(df['prob_bottom'].iloc[t]):
            continue
        move = (close[t] - close[t - TREND_LOOKBACK]) / (close[t - TREND_LOOKBACK] + 1e-10)
        if move < -trend_pct:
            down_probs.append(df['prob_bottom'].iloc[t])
        elif move > trend_pct:
            up_probs.append(df['prob_top'].iloc[t])
        else:
            neutral_probs.append(df['prob_bottom'].iloc[t])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Bottom prob in downtrend vs neutral
    ax = axes[0]
    ax.hist(down_probs, bins=50, alpha=0.6, color='#F44336', density=True,
            label=f'Downtrend (n={len(down_probs)})')
    ax.hist(neutral_probs[:len(down_probs)], bins=50, alpha=0.4, color='gray', density=True,
            label=f'No trend (n={min(len(neutral_probs), len(down_probs))})')
    ax.axvline(x=0.5, color='black', linewidth=1, linestyle='--')
    ax.set_xlabel('P(bottom)')
    ax.set_ylabel('Density')
    ax.set_title('Bottom Probability: Downtrend vs Neutral', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Top prob in uptrend vs neutral
    ax = axes[1]
    ax.hist(up_probs, bins=50, alpha=0.6, color='#2196F3', density=True,
            label=f'Uptrend (n={len(up_probs)})')
    neutral_top = [df['prob_top'].iloc[t] for t in range(TREND_LOOKBACK, len(df))
                   if not np.isnan(df['prob_top'].iloc[t])
                   and abs((close[t] - close[t-TREND_LOOKBACK])/(close[t-TREND_LOOKBACK]+1e-10)) < trend_pct]
    ax.hist(neutral_top[:len(up_probs)], bins=50, alpha=0.4, color='gray', density=True,
            label=f'No trend (n={min(len(neutral_top), len(up_probs))})')
    ax.axvline(x=0.5, color='black', linewidth=1, linestyle='--')
    ax.set_xlabel('P(top)')
    ax.set_ylabel('Density')
    ax.set_title('Top Probability: Uptrend vs Neutral', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.suptitle(f'{stock} — CNN Probability Distribution by Market Context',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
        print(f"  ✓ Saved: {save_path}")
    plt.show()


def plot_precision_by_threshold(df, stock, save_path=None):
    """
    图4: 精确率和信号数量 vs 阈值
    """
    close = df['close'].values
    n = len(close)
    trend_pct = TREND_PCT / 100.0
    LA = CNN_LOOKAHEAD

    thresholds = np.arange(0.3, 0.85, 0.05)
    bot_prec, bot_n = [], []
    top_prec, top_n = [], []

    for thr in thresholds:
        # Bottom precision
        n_correct, n_total = 0, 0
        for t in range(max(CNN_WINDOW, TREND_LOOKBACK), n - LA):
            move = (close[t] - close[t-TREND_LOOKBACK]) / (close[t-TREND_LOOKBACK]+1e-10)
            if move < -trend_pct and df['prob_bottom'].iloc[t] > thr:
                n_total += 1
                # "strong"标签: 未来价格弹幅>0.5%
                if (close[t+LA] - close[t]) / close[t] > trend_pct:
                    n_correct += 1
        bot_prec.append(n_correct / n_total * 100 if n_total > 0 else 0)
        bot_n.append(n_total)

        # Top precision
        n_correct, n_total = 0, 0
        for t in range(max(CNN_WINDOW, TREND_LOOKBACK), n - LA):
            move = (close[t] - close[t-TREND_LOOKBACK]) / (close[t-TREND_LOOKBACK]+1e-10)
            if move > trend_pct and df['prob_top'].iloc[t] > thr:
                n_total += 1
                if (close[t] - close[t+LA]) / close[t] > trend_pct:
                    n_correct += 1
        top_prec.append(n_correct / n_total * 100 if n_total > 0 else 0)
        top_n.append(n_total)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Bottom
    ax1 = axes[0]
    ax2 = ax1.twinx()
    l1 = ax1.plot(thresholds, bot_prec, 'o-', color='#2196F3', linewidth=2, label='Precision (%)')
    l2 = ax2.bar(thresholds, bot_n, width=0.03, alpha=0.3, color='gray', label='N signals')
    ax1.axhline(y=50, color='red', linewidth=1, linestyle='--', alpha=0.5, label='Random (50%)')
    ax1.set_xlabel('Threshold')
    ax1.set_ylabel('Precision (%)', color='#2196F3')
    ax2.set_ylabel('Number of signals', color='gray')
    ax1.set_title('Bottom Detection Precision', fontweight='bold')
    ax1.set_ylim(0, 100)
    lines = l1 + [l2]
    labels = [l.get_label() for l in l1] + ['N signals']
    ax1.legend(l1, [l.get_label() for l in l1], loc='upper left')

    # Top
    ax1 = axes[1]
    ax2 = ax1.twinx()
    l1 = ax1.plot(thresholds, top_prec, 'o-', color='#F44336', linewidth=2, label='Precision (%)')
    l2 = ax2.bar(thresholds, top_n, width=0.03, alpha=0.3, color='gray', label='N signals')
    ax1.axhline(y=50, color='red', linewidth=1, linestyle='--', alpha=0.5)
    ax1.set_xlabel('Threshold')
    ax1.set_ylabel('Precision (%)', color='#F44336')
    ax2.set_ylabel('Number of signals', color='gray')
    ax1.set_title('Top Detection Precision', fontweight='bold')
    ax1.set_ylim(0, 100)
    ax1.legend(l1, [l.get_label() for l in l1], loc='upper left')

    plt.suptitle(f'{stock} — Precision vs Threshold (strong label: >{TREND_PCT}% move)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
        print(f"  ✓ Saved: {save_path}")
    plt.show()


# ============================================================
# MAIN
# ============================================================
print("Loading strong model (5min, best bounce detector)...")

# 尝试加载strong模型
model_path = os.path.join(MODEL_DIR, "cnn_dual_5min_strong.pt")
ckpt = torch.load(model_path, map_location='cpu', weights_only=True)
model = CNNDualModel(n_features=N_PRICE_FEAT, window=CNN_WINDOW).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
sc_mean = ckpt['scaler_mean']
sc_scale = ckpt['scaler_scale']
print(f"  ✓ Loaded from {model_path}")

# 对每只股票生成可视化
for stock in ['AAPL', 'MSFT', 'SPY']:
    print(f"\n{'='*60}")
    print(f"  Generating charts for {stock}")
    print(f"{'='*60}")

    df = load_ticker(stock)
    if df is None:
        print(f"  ✗ Data not found for {stock}")
        continue

    n = len(df)
    test_start = int(n * TRAIN_RATIO)
    df_test = df.iloc[test_start:].reset_index(drop=True)
    print(f"  Test bars: {len(df_test)}")

    # Run inference
    df_test = run_inference(df_test, model, sc_mean, sc_scale)

    # 图1: 价格走势+信号 (选几个有代表性的时间段)
    # 找一个有较多信号的区间
    for start_pct in [0.1, 0.3, 0.5, 0.7]:
        start_idx = int(len(df_test) * start_pct)
        plot_price_with_signals(
            df_test, stock, start_idx, n_bars=500, thr=0.5,
            save_path=os.path.join(SAVE_FIG_DIR, f"{stock}_signals_{int(start_pct*100)}.png")
        )

    # 图2: 信号结果对比
    plot_signal_outcomes(
        df_test, stock,
        save_path=os.path.join(SAVE_FIG_DIR, f"{stock}_outcomes.png")
    )

    # 图3: 概率分布
    plot_probability_distribution(
        df_test, stock,
        save_path=os.path.join(SAVE_FIG_DIR, f"{stock}_prob_dist.png")
    )

    # 图4: 精确率vs阈值
    plot_precision_by_threshold(
        df_test, stock,
        save_path=os.path.join(SAVE_FIG_DIR, f"{stock}_precision.png")
    )

print(f"\n\n{'#'*60}")
print(f"  All figures saved to: {SAVE_FIG_DIR}")
print(f"{'#'*60}")
print(f"""
图表说明:
  *_signals_*.png  — 价格走势 + CNN拐点信号叠加 (4个时间段)
  *_outcomes.png   — 信号触发后的价格轨迹 (真反转 vs 假反弹)
  *_prob_dist.png  — CNN概率在不同市场状态下的分布
  *_precision.png  — 精确率随阈值变化 (可用于选择最优阈值)

论文使用建议:
  - signals图展示CNN的实际预测效果
  - outcomes图解释为什么81%检测准确率≠方向预测能力
  - precision图支持"CNN能识别反弹形态"的结论
""")